# 🕸️ Long-Term Multi-Person Action Recognition and Tracking

---

> **Research**: MSc Artificial Intelligence & Data Science University of Hull  
> **Student**: Amos Okpe  
> **Student ID**: 202536977  
> **Supervisor**: Dr Vincent Zakka  
> **Model**: AlphAction (SlowFast ResNet-101) - RE-ID & Face Recognition 
> **Environment**: NVIDIA A100 GPU · CUDA 12.3 · PyTorch 2.x · Python 3.9 · Kernel: `alphaction_a100`

 - 👉 Most of the code here can also run from the terminal and available on github.

---

<p align="center">
  <img src="https://res.cloudinary.com/dhqdentd8/image/upload/v1772727050/ideogram-v3.0_a_cover_image_for_my_msc_project_Long-Term_Multi-Person_Action_Recognition_and_T-0_s5gty7.png" width="100%" alt="Cover"/>
</p>

## ❄️ Executive Summary

This notebook contains the complete implementation, execution, and evaluation framework for the AlphAction long-term multi-person action recognition and tracking pipeline.

# 👉 **Clone the repo**
Uncomment and run the line below if you have not yet cloned the repo.

In [ ]:
# !git clone https://github.com/PythonDecorator/Long-Term-Multi-Person-Action-Recognition-and-Tracking

## ⚙️ Install Dependencies
A conda environment alphaction_a100 have been created and this notebook runs on it, so we do not need to install the packages here any more, please create a conda env if you need to run this notebook as it depend on C++ and we had to forge some packages. <br>
See `README.MD` on github on how to create the env.

# 👉 **Imports, GPU Verification & Global Constants**

## ⚙️ Silent HPC warring logs
There were some warrings from the HPC env.
Those pthread_setaffinity_np errors are harmless, they just mean the HPC scheduler won't let onnxruntime pin threads to specific CPU cores. 

In [ ]:
import os
import logging
import warnings

os.environ["KMP_AFFINITY"] = "disabled"
os.environ["OMP_NUM_THREADS"] = "8" # Explicitly setting threads stops the warning

warnings.filterwarnings("ignore")

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'          
os.environ['ONNXRUNTIME_MIN_LOG_LEVEL'] = '3'     
os.environ['OPENCV_LOG_LEVEL'] = 'ERROR'          

loggers_to_mute = ["ultralytics", "insightface", "matplotlib", "PIL", "urllib3"]
for logger_name in loggers_to_mute:
    logging.getLogger(logger_name).setLevel(logging.ERROR)

try:
    import onnxruntime as ort
    ort.set_default_logger_severity(4) # 3 = Error, 4 = Fatal Only
except ImportError:
    pass

print("🤫 Silent Mode Activated: ONNX thread affinity errors suppressed.")

## ⚙️ **General Imports**

In [ ]:
import sys
import gc
import time
import argparse
import subprocess
from pathlib import Path
from datetime import datetime
from base64 import b64encode
from itertools import count as _count
from time import sleep as _sleep
from collections import Counter, defaultdict, deque
from dataclasses import dataclass, field
from enum import Enum, auto

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics.pairwise import cosine_similarity

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.templates.default = "plotly_white"

from IPython.display import display, HTML
from tqdm import tqdm

import cv2

import torch
import torch.nn as _nn
import torch.nn.functional as F
import torchvision
import torchvision.models as tvm
import torchvision.transforms as tvt

try:
    from insightface.app import FaceAnalysis
    _INSIGHTFACE_OK = True
    print("✅ InsightFace available")
except ImportError:
    _INSIGHTFACE_OK = False
    print("❌  InsightFace not installed — Section 3 face recognition cells will be skipped")
    print("   Install: pip install insightface onnxruntime-gpu")

print()
print("✅ Core imports successful")
print(f"  Python     : {sys.version.split()[0]}")
print(f"  NumPy      : {np.__version__}")
print(f"  OpenCV     : {cv2.__version__}")
print(f"  PyTorch    : {torch.__version__}")
print(f"  TorchVision: {torchvision.__version__}")

## ⚙️ **GPU VERIFICATION**

In [ ]:
def perform_gpu_test():
    print("=" * 58)
    print("  GPU / CUDA VERIFICATION")
    print("=" * 58)
    cuda_available = torch.cuda.is_available()
    print(f"  CUDA available : {cuda_available}")
    if cuda_available:
        device = torch.device("cuda")
        gpu_name   = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  GPU name       : {gpu_name} ✅")
        print(f"  VRAM           : {gpu_memory:.1f} GB")
        print(f"  CUDA version   : {torch.version.cuda}")
        print(f"  cuDNN          : {torch.backends.cudnn.version()}")
        x = torch.randn(1000, 1000, device=device)
        y = torch.matmul(x, x.T)
        print(f"  Smoke test     : ✓ 1000×1000 matmul → {y.shape}")
        del x, y; torch.cuda.empty_cache()
    else:
        device = torch.device("cpu")
        print("  ⚠  No GPU — running on CPU (expect slower inference)")
    print(f"  Active device  : {device}")
    print("=" * 58)
    return device

device = perform_gpu_test()

## ⚙️ **GLOBAL CONSTANTS**

In [ ]:
PROJECT_ROOT = Path("Long-Term-Multi-Person-Action-Recognition-and-Tracking/AlphactionFramework").resolve()
DEMO_DIR     = PROJECT_ROOT / "demo"
CONFIG_DIR   = PROJECT_ROOT / "config_files"
WEIGHTS_DIR  = PROJECT_ROOT / "data" / "models" / "aia_models"
RESULTS_DIR  = DEMO_DIR / "test_results"
FIGURES_DIR  = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

TEST_VIDEO = "test_video_3s_480p.mp4"

MODEL_CFG     = CONFIG_DIR / "resnet101_8x8f_denseserial.yaml"
MODEL_WEIGHTS = WEIGHTS_DIR / "resnet101_8x8f_denseserial.pth"

CONFIDENCE_THRESHOLD = 0.5
MC_DROPOUT_PASSES    = 30
BOOTSTRAP_N          = 10_000
RANDOM_SEED          = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

SAVE_PLOTS = True
PLOT_COLORS = {
    'red': '#FF0000',
    'green': '#00FF00',
    'primary': '#007bff',
    'secondary': '#6c757d',
    'accent': '#ffc107',
    'gray': '#6c757d',
    'dark': '#343a40'
}
FONT_SIZE = 14

print("✅ Constants configured")
print(f"  Project root  : {PROJECT_ROOT}")
print(f"  Model config  : {MODEL_CFG.name} — exists: {MODEL_CFG.exists()}")
print(f"  Model weights : {MODEL_WEIGHTS.name} — exists: {MODEL_WEIGHTS.exists()}")
print(f"  Figures dir   : {FIGURES_DIR}")

# ⚙️ NOTEBOOK HELPER CLASS

In [ ]:
class NotebookUtils:
    @staticmethod
    def show_video(video_path: str, width: int = 800) -> None:
        if not os.path.exists(video_path):
            print(f"❌ Error: Video not found at '{video_path}'")
            return

        h264_path = video_path.replace(".mp4", "_h264.mp4")
        os.system(f"ffmpeg -y -i '{video_path}' -vcodec libx264 -pix_fmt yuv420p -loglevel quiet '{h264_path}'")
        play_path = h264_path if os.path.exists(h264_path) else video_path
        
        with open(play_path, "rb") as video_file:
            b64_video = b64encode(video_file.read()).decode()
            
        html_code = f'''
            <video width="{width}" controls autoplay loop style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.15); border: 1px solid #ddd;">
                <source src="data:video/mp4;base64,{b64_video}" type="video/mp4">
            </video>
        '''
        display(HTML(html_code))

    @staticmethod
    def save_plot(fig, name: str, folder: str = "plots", dpi: int = 300, width: int = 1200, height: int = 800) -> str:
        os.makedirs(folder, exist_ok=True)
        scale = dpi / 96  # Scale factor for target DPI
        
        png_filename = os.path.join(folder, f"{name}.png")
        html_filename = os.path.join(folder, f"{name}_failsafe.html")
        
        try:
            fig.write_image(png_filename, scale=scale, width=width, height=height)
            print(f"📸 SUCCESS: Saved high-res plot ({dpi} DPI) to: {png_filename}")
            return png_filename
            
        except Exception as e:
            print(f"⚠️ Kaleido PNG export failed on this server. Proceeding to HTML failsafe...")
            
            config = {
              'toImageButtonOptions': {
                'format': 'png', 
                'filename': name,
                'height': height,
                'width': width,
                'scale': scale # This ensures the camera button downloads at 300 DPI!
              }
            }
            
            fig.write_html(html_filename, config=config)
            print(f"✅ FAILSAFE SUCCESS: Saved interactive HTML to: {html_filename}")
            print(f"   -> Download '{html_filename}' to your computer.")
            print(f"   -> Open it in your web browser (Chrome/Edge/Safari).")
            print(f"   -> Hover over the chart and click the 'Camera' icon. It will download your 300 DPI PNG!")
            
            return html_filename

    @staticmethod
    def compare_videos(
        videos:             list,
        n_frames:           int   = 5,
        title:              str   = None,
        row_labels:         list  = None,
        row_colors:         list  = None,
        dpi:                int   = 130,
        seed:               int   = 42,
        fig_height_per_row: float = 2.8,
    ) -> None:
        import io as _io
        import random as _random
        import cv2 as _cv2_cmp
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as _plt
        import matplotlib.gridspec as _gridspec
        from pathlib import Path as _Path
        from IPython.display import display as _disp, Image as _Img

        BG = "#0d0d1a"
        _DEF_LABELS_3 = [
            "BEFORE  Re-ID  (raw tracker IDs)",
            "WITH  Fine-tuning  (domain-adapted)",
            "WITHOUT  Fine-tuning  (ImageNet)",
        ]
        _DEF_LABELS_2 = ["BEFORE  Re-ID  (raw video)", "Annotated Output"]
        _DEF_COLORS   = ["#ffffff", "#caffbf", "#a0c4ff"]

        if not (2 <= len(videos) <= 3):
            print("❌  compare_videos() requires 2 or 3 video paths."); return

        n_rows = len(videos)
        parsed = []
        for i, entry in enumerate(videos):
            d = entry if isinstance(entry, dict) else {"path": str(entry)}
            d.setdefault("label", (row_labels[i] if row_labels and i < len(row_labels)
                                   else (_DEF_LABELS_3 if n_rows == 3 else _DEF_LABELS_2)[i]))
            d.setdefault("color", (row_colors[i] if row_colors and i < len(row_colors)
                                   else _DEF_COLORS[i]))
            parsed.append(d)

        for p in parsed:
            if not _Path(p["path"]).exists():
                print(f"❌  Not found: {p['path']}"); return

        def _get_info(path):
            cap = _cv2_cmp.VideoCapture(path)
            n   = int(cap.get(_cv2_cmp.CAP_PROP_FRAME_COUNT))
            fps = cap.get(_cv2_cmp.CAP_PROP_FPS) or 25
            cap.release()
            return n, fps

        infos  = [_get_info(p["path"]) for p in parsed]
        total  = min(n for n, _ in infos)
        fps    = infos[0][1]

        if total < 2:
            print(f"❌  Video too short ({total} frames)."); return

        rng = _random.Random(seed)
        lo  = int(total * 0.10)
        hi  = int(total * 0.90)
        if hi - lo <= n_frames:
            indices = list(range(lo, hi))
        else:
            step    = (hi - lo) / n_frames
            indices = []
            for i in range(n_frames):
                base = int(lo + i * step)
                jit  = rng.randint(-max(1, int(step * 0.15)),
                                    max(1, int(step * 0.15)))
                indices.append(max(lo, min(hi - 1, base + jit)))
            indices = sorted(set(indices))

        def _extract(path, idxs):
            cap, out = _cv2_cmp.VideoCapture(path), {}
            for idx in idxs:
                cap.set(_cv2_cmp.CAP_PROP_POS_FRAMES, idx)
                ok, fr = cap.read()
                if ok:
                    out[idx] = _cv2_cmp.cvtColor(fr, _cv2_cmp.COLOR_BGR2RGB)
            cap.release()
            return [out[i] for i in idxs if i in out]

        all_imgs = [_extract(p["path"], indices) for p in parsed]
        n_cols   = min(len(imgs) for imgs in all_imgs)
        if n_cols == 0:
            print("❌  Could not extract frames."); return

        fig_w = max(4 * n_cols, 12)
        fig_h = fig_height_per_row * n_rows + 0.6
        fig   = _plt.figure(figsize=(fig_w, fig_h), facecolor=BG)
        fig.suptitle(
            f"📹  {title or _Path(parsed[0]['path']).stem}",
            fontsize=14, fontweight="bold", color="white", y=0.99,
        )
        top_pad    = 1.0 - (0.45 / fig_h)
        bottom_pad = 0.55 / fig_h
        gs = _gridspec.GridSpec(
            n_rows, n_cols, figure=fig,
            hspace=0.04, wspace=0.03,
            top=top_pad, bottom=bottom_pad,
            left=0.08, right=0.995,
        )

        for row_i, (imgs, meta) in enumerate(zip(all_imgs, parsed)):
            lc = meta["color"]
            for col_i, img in enumerate(imgs[:n_cols]):
                ax = fig.add_subplot(gs[row_i, col_i])
                ax.imshow(img, aspect="auto")
                ax.set_facecolor(BG)
                ax.axis("off")
                for sp in ax.spines.values():
                    sp.set_visible(True); sp.set_edgecolor(lc)
                    sp.set_linewidth(1.4 if col_i == 0 else 0.7); sp.set_alpha(0.7)
                fn = indices[col_i]; ts = fn / fps
                ax.text(0.02, 0.97,
                        f"#{fn}  {int(ts//60):02d}:{ts%60:05.2f}",
                        transform=ax.transAxes, fontsize=6.5,
                        color="white", va="top", ha="left",
                        bbox=dict(boxstyle="round,pad=0.25",
                                  facecolor="#000000", alpha=0.60, edgecolor="none"))
                if col_i == 0:
                    ax.text(-0.06, 0.5, meta["label"],
                            transform=ax.transAxes,
                            fontsize=9.0, fontweight="bold", color=lc,
                            va="center", ha="right",
                            rotation=90, rotation_mode="anchor")

        buf = _io.BytesIO()
        fig.savefig(buf, format="png", dpi=dpi,
                    bbox_inches="tight", facecolor=BG)
        _plt.close(fig)
        buf.seek(0)
        _disp(_Img(data=buf.read()))
        print(f"✅  {n_rows} rows × {n_cols} frames displayed.")

    @staticmethod
    def compare_condition_results(
        results_dir:   str  = "test_results",
        raw_video_dir: str  = "test_videos",
        n_frames:      int  = 1,
        conditions:    list = None,
        dpi:           int  = 130,
    ) -> None:
        from pathlib import Path as _Path

        VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}
        res_root   = _Path(results_dir)
        raw_root   = _Path(raw_video_dir)

        if conditions is None:
            conditions = sorted(
                d.name for d in res_root.iterdir()
                if d.is_dir() and not d.name.startswith(".")
            ) if res_root.exists() else []

        if not conditions:
            print(f"❌  No condition folders found in '{results_dir}'."); return

        print(f"\n📊  Condition results overview  —  {len(conditions)} conditions")
        print("=" * 60)

        for cond in conditions:
            res_dir = res_root / cond
            raw_dir = raw_root / cond

            out_videos = sorted(
                p for p in res_dir.iterdir()
                if p.suffix.lower() in VIDEO_EXTS and "_h264" not in p.name
            ) if res_dir.exists() else []

            if not out_videos:
                print(f"  [{cond}]  no output videos — skipping."); continue

            for out_vid in out_videos:
                stem    = out_vid.stem.replace("_output", "")
                raw_vid = None
                if raw_dir.exists():
                    for ext in VIDEO_EXTS:
                        cand = raw_dir / (stem + ext)
                        if cand.exists():
                            raw_vid = cand; break

                cond_label = cond.replace("_", " ").title()
                if raw_vid is None:
                    vids   = [str(out_vid), str(out_vid)]
                    labels = [f"[{cond_label}]  {stem}  — output (no raw found)",
                              f"[{cond_label}]  {stem}  — output"]
                else:
                    vids   = [str(raw_vid), str(out_vid)]
                    labels = [f"[{cond_label}]  {stem}  — raw",
                              f"[{cond_label}]  {stem}  — annotated"]

                NotebookUtils.compare_videos(
                    videos     = vids,
                    n_frames   = n_frames,
                    title      = f"[{cond_label}]  {stem}",
                    row_labels = labels,
                    row_colors = ["#ffffff", "#caffbf"],
                    dpi        = dpi,
                )

    @staticmethod
    def cleanup_gpu():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print("🧹 GPU Memory cleared.")

print("✅ NotebookUtils Class Ready.")

# ❄️ **The AlphAction Framework**

This project extends the AlphAction framework (Tang et al., 2020) for long-term multi-person behaviour understanding.
The framework combines a SlowFast ResNet-101 backbone with an Interaction Aggregation module and a dynamic memory pool to model spatio-temporal interactions.

## Refactoring for Modern HPC Environments

The original 2020 implementation (Python 3.7 / PyTorch 1.4) required substantial refactored for the NVIDIA A100 environment:

## Training & Evaluation Scripts - `train_net.py` & `test_net.py`

These two scripts are the standard AlphAction training and evaluation entry-points for the **AVA dataset**.  
They are not run inside this notebook (they require the full distributed AVA dataset and multi-GPU setup).  

- **`train_net.py`**: Full distributed training on AVA.  Supports transfer learning (`--transfer`), LR adjustment (`--adjust-lr`), TensorBoard logging (`--use-tfboard`).
- **`test_net.py`**:  Runs inference on the AVA test set and writes per-video metrics to `cfg.OUTPUT_DIR/inference/`.

The pretrained weights we use (`resnet101_8x8f_denseserial.pth`) were produced by running `train_net.py`  
on the full AVA v2.2 dataset with the config `resnet101_8x8f_denseserial.yaml`.


In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(DEMO_DIR))

try:
    import alphaction
    from alphaction.config import cfg
    from alphaction.modeling.detector import build_detection_model
    from alphaction.structures.bounding_box import BoxList
    from alphaction.structures.memory_pool import MemoryPool
    from alphaction.layers import FrozenBatchNorm3d, ROIAlign3d
    print("✅ AlphAction package imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Fix: cd AlphactionFramework && FORCE_CUDA=1 pip install -e .")

## ⚙️ **LOAD MODEL AND VERIFY WEIGHTS**

In [ ]:
print("Loading AlphActionFramework model...")
if not MODEL_CFG.exists():
    print("  ❌  Config not found — skipping model load: PLEASE UPLOAD")
    model = None
elif not MODEL_WEIGHTS.exists():
    print("  ❌  Weights not found — skipping model load, PLEASE UPLOAD THE MODELS")
    model = None
else:
    try:
        cfg.merge_from_file(str(MODEL_CFG))
        cfg.freeze()
        model = build_detection_model(cfg)
        checkpoint = torch.load(str(MODEL_WEIGHTS), map_location="cpu")
        model.load_state_dict(checkpoint.pop("model"), strict=False)
        model = model.to(device)
        model.eval()

        n_params = sum(p.numel() for p in model.parameters())
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  ✅ Model loaded successfully")
        print(f"  Parameters (total)    : {n_params:,}")
        print(f"  Parameters (trainable): {n_trainable:,}")
        print(f"  Memory on GPU         : {torch.cuda.memory_allocated()/1e6:.1f} MB")
    except Exception as e:
        print(f"  ❌ Model load error: {e}")
        model = None

# 👨‍💻 Lets see what is in our working directory (root)

In [ ]:
!ls

# 👨‍💻Pipeline Classes
The four core files (`video_detection_loader.py`, `action_predictor.py`, `visualizer.py`, `demo.py`)   

`VideoDetectionLoader` Reads frames from a video file or webcam, runs YOLO person detection <br>
tracking, and feeds results into two queues consumed by downstream workers.


In [ ]:
import queue
import threading
from itertools import count as _count
from time import sleep as _sleep
from typing import Optional, Tuple

import cv2 as _cv2
import numpy as _np
import torch as _torch
from torchvision.transforms import functional as _TF
from tqdm import tqdm as _tqdm

from detector.apis import get_detector as _get_detector

class SharedValue:
    def __init__(self, initial: int = 0) -> None:
        self._value = initial
        self._lock  = threading.Lock()

    @property
    def value(self) -> int:
        with self._lock:
            return self._value

    @value.setter
    def value(self, v: int) -> None:
        with self._lock:
            self._value = v

    def _drain(q: queue.Queue) -> None:
        while True:
            try:
                q.get_nowait()
            except queue.Empty:
                break

class VideoDetectionLoader:
    def __init__(self, cfg, track_queue, action_queue, predictor_process) -> None:
        self.cfg         = cfg
        self.input_path  = cfg.input_path
        self.start_ms    = cfg.start
        self.dur_ms      = cfg.duration
        self.realtime    = cfg.realtime
        self.detector    = None   # lazy — created inside worker thread

        cap = _cv2.VideoCapture(self.input_path)
        assert cap.isOpened(), f"Cannot open: {self.input_path}"
        self.videoinfo = {
            "fourcc":    int(cap.get(_cv2.CAP_PROP_FOURCC)),
            "fps":       cap.get(_cv2.CAP_PROP_FPS),
            "frameSize": (int(cap.get(_cv2.CAP_PROP_FRAME_WIDTH)),
                          int(cap.get(_cv2.CAP_PROP_FRAME_HEIGHT))),
        }
        cap.release()

        self._stop        = threading.Event()
        self.track_queue  = track_queue
        self.action_queue = action_queue
        self.predictor    = predictor_process

    def start(self) -> "VideoDetectionLoader":
        self._worker = threading.Thread(
            target=self._run, daemon=True, name="VDL-worker")
        self._worker.start()
        return self

    def terminate(self) -> None:
        self._stop.set()
        self._worker.join(timeout=10)
        _drain(self.track_queue)
        _drain(self.action_queue)

    def read_track(self):
        return self.track_queue.get()

    @property
    def stopped(self) -> bool:
        return self._stop.is_set()

    def _run(self) -> None:
        _cv2.setNumThreads(0)
        _torch.set_num_threads(1)
        self.detector = _get_detector(self.cfg)

        cap = _cv2.VideoCapture(self.input_path)
        assert cap.isOpened()
        if not self.realtime:
            cap.set(_cv2.CAP_PROP_POS_MSEC, self.start_ms)

        prev_ms = 0.0
        for frame_i in _count():
            if self._stop.is_set():
                break
            if self.realtime and self.predictor.value == -1:
                break
            if self.track_queue.full():
                _sleep(0.001)
                continue

            ok, frame = cap.read()
            prev_ms = cur_ms = cap.get(_cv2.CAP_PROP_POS_MSEC)

            if not ok or (not self.realtime and self.dur_ms != -1
                          and cur_ms > self.start_ms + self.dur_ms):
                self._put(self.track_queue,  (None, None, None, None))
                self._put(self.action_queue, ("Done", self.videoinfo["frameSize"]))
                self._await_predictor(prev_ms)
                break

            img_t    = self.detector.image_preprocess(frame)
            if isinstance(img_t, _np.ndarray):
                img_t = _torch.from_numpy(img_t)
            if img_t.dim() == 3:
                img_t = img_t.unsqueeze(0)

            orig_img = frame[:, :, ::-1]
            im_dim   = _torch.FloatTensor(
                (frame.shape[1], frame.shape[0])).repeat(1, 2)

            with _torch.no_grad():
                det = self._detect(img_t, orig_img, im_dim)
            self._postprocess(det, frame, cur_ms, frame_i)

        cap.release()

    def _detect(self, img, orig_img, im_dim):
        with _torch.no_grad():
            dets = self.detector.images_detection(img, im_dim)
        if isinstance(dets, int) or dets.shape[0] == 0:
            return orig_img, None, None, None
        if isinstance(dets, _np.ndarray):
            dets = _torch.from_numpy(dets)
        dets  = dets.cpu()
        mask  = dets[:, 0] == 0
        boxes = dets[mask, 1:5]
        scores= dets[mask, 5:6]
        ids   = dets[mask, 6:7]
        if boxes.shape[0] == 0:
            return orig_img, None, None, None
        return orig_img, boxes, scores, ids

    def _postprocess(self, detection, raw_frame, cur_ms, frame_i) -> None:
        orig_img, boxes, scores, ids = detection
        if orig_img is None or self._stop.is_set():
            self._put(self.track_queue, (None, None, None, None))
            return
        self._put(self.action_queue,
                  ((raw_frame, cur_ms, boxes, scores, ids),
                   self.videoinfo["frameSize"]))
        self._put(self.track_queue, (orig_img, boxes, scores, ids))

    def _put(self, q, item) -> None:
        if not self._stop.is_set():
            q.put(item)

    def _await_predictor(self, last_ms: float) -> None:
        pred_val = self.predictor.value
        if pred_val == -1:
            return
        _tqdm.write("Tracking done. Waiting for feature extraction…")
        initial  = max(pred_val - self.start_ms, 0)
        total    = max(int(last_ms) - self.start_ms, 1)
        pbar     = _tqdm(total=total, initial=initial, desc="Feature Extraction")
        last_pos = initial
        while self.predictor.value != -1:
            cur = self.predictor.value
            pbar.update(cur - last_pos)
            last_pos = cur
            if self._stop.is_set():
                break
            _sleep(0.1)
        pbar.update(total - last_pos)
        pbar.close()

print("✅ VideoDetectionLoader  ready")

In [ ]:
import copy
import queue
import threading
from bisect import bisect_right
from itertools import count as _count
from typing import Any, Dict, List, Optional, Tuple

import numpy as _np
import torch as _torch
from tqdm import tqdm as _tqdm

from alphaction.config import cfg as _base_cfg
from alphaction.dataset.collate_batch import batch_different_videos
from alphaction.dataset.transforms import object_transforms as OT
from alphaction.dataset.transforms import video_transforms as T
from alphaction.modeling.detector import build_detection_model
from alphaction.structures.bounding_box import BoxList
from alphaction.structures.memory_pool import MemoryPool
from alphaction.utils.IA_helper import has_memory, has_object
from alphaction.utils.checkpoint import ActionCheckpointer

def _to_device(v: Any, device: _torch.device) -> Any:
    if isinstance(v, _torch.Tensor):
        return v.to(device).float()
    if isinstance(v, BoxList):
        bl = BoxList(v.bbox.to(device).float(), v.size, v.mode)
        for name, val in v.extra_fields.items():
            if isinstance(val, _torch.Tensor):
                bl.add_field(name, val.to(device).float())
            elif hasattr(val, "to"):
                bl.add_field(name, val.to(device))
            else:
                bl.add_field(name, val)
        return bl
    if hasattr(v, "to"):
        return v.to(device)
    return v

class AVAPredictor:
    def __init__(self, cfg_path, weight_path, detect_rate,
                 common_cate, device) -> None:
        cfg = _base_cfg.clone()
        cfg.merge_from_file(cfg_path)
        cfg.MODEL.WEIGHT = weight_path
        cfg.MODEL.IA_STRUCTURE.MEMORY_RATE *= detect_rate
        if common_cate:
            cfg.MODEL.ROI_ACTION_HEAD.NUM_CLASSES                     = 15
            cfg.MODEL.ROI_ACTION_HEAD.NUM_PERSON_MOVEMENT_CLASSES     = 6
            cfg.MODEL.ROI_ACTION_HEAD.NUM_OBJECT_MANIPULATION_CLASSES = 5
            cfg.MODEL.ROI_ACTION_HEAD.NUM_PERSON_INTERACTION_CLASSES  = 4
        cfg.freeze()
        self.cfg        = cfg
        self.device     = device
        self.cpu_device = _torch.device("cpu")

        self.has_memory = has_memory(cfg.MODEL.IA_STRUCTURE)
        self.mem_len    = cfg.MODEL.IA_STRUCTURE.LENGTH
        self.mem_rate   = cfg.MODEL.IA_STRUCTURE.MEMORY_RATE
        self.has_object = has_object(cfg.MODEL.IA_STRUCTURE)

        self.mem_pool      = MemoryPool()
        self.object_pool   = MemoryPool()
        self.mem_timestamps: List[int] = []
        self.obj_timestamps: List[int] = []
        self.pred_pos = 0
        self.model    = None   # lazy init

        self.transforms, self.person_transforms, self.object_transforms = (
            self._build_transforms()
        )

    def ensure_model(self) -> None:
        if self.model is not None:
            return
        self.model = build_detection_model(self.cfg)
        self.model.eval()
        ckpt = ActionCheckpointer(self.cfg, self.model)
        print(f"Loading weights: {self.cfg.MODEL.WEIGHT}")
        ckpt.load(self.cfg.MODEL.WEIGHT)
        self.model.to(self.device)
        print("Model ready.")

    def _build_transforms(self):
        cfg = self.cfg
        video_tf = T.Compose([
            T.TemporalCrop(cfg.INPUT.FRAME_NUM, cfg.INPUT.FRAME_SAMPLE_RATE),
            T.Resize(cfg.INPUT.MIN_SIZE_TEST, cfg.INPUT.MAX_SIZE_TEST),
            T.ToTensor(),
            T.Normalize(mean=cfg.INPUT.PIXEL_MEAN, std=cfg.INPUT.PIXEL_STD,
                        to_bgr=cfg.INPUT.TO_BGR),
            T.SlowFastCrop(cfg.INPUT.TAU, cfg.INPUT.ALPHA, False),
        ])
        person_tf = OT.Resize()
        object_tf = OT.Compose([
            OT.PickTop(cfg.MODEL.IA_STRUCTURE.MAX_OBJECT),
            OT.Resize(),
        ])
        return video_tf, person_tf, object_tf

    def update_feature(self, video_data, boxes, objects, timestamp, rands):
        self.ensure_model()
        if self.mem_timestamps:
            assert timestamp > self.mem_timestamps[-1]

        slow = batch_different_videos(
            [video_data[0]], self.cfg.DATALOADER.SIZE_DIVISIBILITY
        ).to(self.device)
        fast = batch_different_videos(
            [video_data[1]], self.cfg.DATALOADER.SIZE_DIVISIBILITY
        ).to(self.device)
        boxes_gpu = [self.person_transforms(boxes, rands).to(self.device)]
        objs_gpu  = (
            [self.object_transforms(objects, rands).to(self.device)]
            if objects is not None else None
        )
        with _torch.no_grad():
            feat   = self.model(slow, fast, boxes_gpu, objs_gpu, part_forward=0)
            p_feat = [f.to(self.cpu_device) for f in feat[0]][0]
            o_feat = (None if feat[1] is None
                      else [f.to(self.cpu_device) for f in feat[1]][0])

        self.mem_pool["SingleVideo", timestamp] = p_feat
        self.mem_timestamps.append(timestamp)
        if o_feat is not None:
            self.object_pool["SingleVideo", timestamp] = o_feat
            self.obj_timestamps.append(timestamp)

    def n_ready(self) -> int:
        if not self.mem_timestamps:
            return 0
        if self.has_memory:
            before, after = self.mem_len
            last_ready = self.mem_timestamps[-1] - after * self.mem_rate
            return bisect_right(self.mem_timestamps, last_ready) - self.pred_pos
        return len(self.mem_timestamps) - self.pred_pos

    def predict(self, timestamp: int, vid_size: Tuple[int, int]):
        self.ensure_model()
        td = lambda v: _to_device(v, self.device)
        current_p = [td(self.mem_pool["SingleVideo", timestamp])]
        current_o = (
            [td(self.object_pool["SingleVideo", timestamp])]
            if ("SingleVideo", timestamp) in self.object_pool else None
        )
        gpu_pool = MemoryPool()
        for k, v in self.mem_pool.items():
            gpu_pool[k] = td(v)
        extras = dict(
            person_pool    = gpu_pool,
            movie_ids      = ["SingleVideo"],
            timestamps     = [timestamp],
            current_feat_p = current_p,
            current_feat_o = current_o,
        )
        with _torch.no_grad():
            out = self.model(None, None, None, None, extras=extras, part_forward=1)
            out = [o.resize(vid_size).to(self.cpu_device) for o in out]
        self.pred_pos += 1
        return out[0]

    def release_feature(self, timestamp: Optional[int] = None) -> None:
        if timestamp is None:
            self.mem_pool = MemoryPool(); self.object_pool = MemoryPool()
            self.mem_timestamps = []; self.obj_timestamps = []
            self.pred_pos = 0
            return
        last_unused = (timestamp - self.mem_len[0] * self.mem_rate
                       if self.has_memory else timestamp)
        n = bisect_right(self.mem_timestamps, last_unused)
        for t in self.mem_timestamps[:n]:
            del self.mem_pool["SingleVideo", t]
        self.mem_timestamps = self.mem_timestamps[n:]
        self.pred_pos = max(self.pred_pos - n, 0)
        m = bisect_right(self.obj_timestamps, timestamp)
        for t in self.obj_timestamps[:m]:
            del self.object_pool["SingleVideo", t]
        self.obj_timestamps = self.obj_timestamps[m:]

class AVAPredictorWorker:
    def __init__(self, cfg) -> None:
        self.realtime    = cfg.realtime
        self.detect_rate = cfg.detect_rate
        self.interval    = 1000 // cfg.detect_rate

        self.predictor = AVAPredictor(
            cfg_path    = cfg.cfg_path,
            weight_path = cfg.weight_path,
            detect_rate = cfg.detect_rate,
            common_cate = cfg.common_cate,
            device      = cfg.device,
        )

        self.obj_det  = None
        self._obj_cfg = None
        if self.predictor.has_object:
            obj_cfg          = copy.deepcopy(cfg)
            obj_cfg.detector = "yolo"
            self._obj_cfg    = obj_cfg

        self.track_queue  = queue.Queue(maxsize=1)
        self.input_queue  = queue.Queue(maxsize=30)
        self.output_queue = queue.Queue()
        self._pred_pos    = SharedValue(0)

        VideoDetectionLoader(
            cfg, self.track_queue, self.input_queue, self._pred_pos
        ).start()

        ava_cfg           = self.predictor.cfg
        self._buf_len     = ava_cfg.INPUT.FRAME_NUM * ava_cfg.INPUT.FRAME_SAMPLE_RATE
        self._center      = self._buf_len // 2
        self._frames: List      = []
        self._extras: List      = []
        self._timestamps: List  = []
        self._last_ms     = -2000
        self._pred_cnt    = 0
        self._vid_tf      = self.predictor.transforms

        self._stop_event    = threading.Event()
        self._tracking_done = threading.Event()

        self._thread = threading.Thread(
            target=self._predict_loop, daemon=True, name="Predictor")
        self._thread.start()

    def read_track(self):
        return self.track_queue.get()

    def read_result(self):
        if self._stop_event.is_set():
            return None
        try:
            return self.output_queue.get_nowait()
        except queue.Empty:
            return None

    def signal_tracking_done(self) -> None:
        assert not self.realtime
        self._tracking_done.set()

    def terminate(self) -> None:
        self._stop_event.set()
        self._pred_pos.value = -1
        self._thread.join(timeout=15)
        _drain(self.input_queue)

    def _predict_loop(self) -> None:
        import cv2 as _cv2_inner
        _cv2_inner.setNumThreads(0)
        _torch.set_num_threads(1)

        if self._obj_cfg is not None:
            self.obj_det = _get_detector(self._obj_cfg)

        stream_done = False
        pred_cnt    = 0

        for _ in _count():
            if self._stop_event.is_set():
                return

            if self._tracking_done.is_set() and stream_done:
                _tqdm.write("Flushing remaining action predictions…")
                remaining = self._timestamps[pred_cnt:]
                for ts, vid_size, ids in _tqdm(
                    remaining, initial=pred_cnt,
                    total=len(self._timestamps), desc="Action Prediction",
                ):
                    feat_idx = ts // self.interval
                    preds    = self.predictor.predict(feat_idx, vid_size)
                    self.output_queue.put((preds, ts, ids))
                    self.predictor.release_feature(feat_idx)
                self.predictor.release_feature()
                _tqdm.write("Action prediction complete.")
                self.output_queue.put("done")
                self._tracking_done.clear()
                return

            try:
                extra, vid_size = self.input_queue.get(timeout=1)
            except queue.Empty:
                continue

            if extra == "Done":
                stream_done          = True
                self._pred_pos.value = -1
                continue

            frame, cur_ms, boxes, scores, ids = extra
            self._frames.append(frame)
            self._extras.append((cur_ms, boxes, scores, ids))
            self._frames = self._frames[-self._buf_len:]
            self._extras = self._extras[-self._buf_len:]

            if len(self._frames) < self._buf_len:
                continue

            ctr_ms, p_boxes, p_scores, p_ids = self._extras[self._center]
            if ctr_ms < self._last_ms + self.interval:
                continue
            self._last_ms = ctr_ms

            if not self.realtime:
                self._pred_pos.value = int(cur_ms)

            if p_boxes is None or len(p_boxes) == 0:
                continue

            frame_arr = _np.stack(self._frames)[..., ::-1]
            obj_boxes = self._detect_objects(vid_size)
            vid_data, _, rands = self._vid_tf(frame_arr, None)
            p_box    = BoxList(p_boxes, vid_size, "xyxy").clip_to_image()
            feat_idx = int(ctr_ms) // self.interval

            self.predictor.update_feature(vid_data, p_box, obj_boxes, feat_idx, rands)

            if self.realtime:
                preds = self.predictor.predict(feat_idx, vid_size)
                self.output_queue.put((preds, ctr_ms, p_ids[:, 0]))
                self.predictor.release_feature(feat_idx)
                pred_cnt += 1
            else:
                self._timestamps.append((int(ctr_ms), vid_size, p_ids[:, 0]))
                n_ready = self.predictor.n_ready()
                for i in range(pred_cnt, pred_cnt + n_ready):
                    ts, vs, ts_ids = self._timestamps[i]
                    fi   = ts // self.interval
                    pred = self.predictor.predict(fi, vs)
                    self.output_queue.put((pred, ts, ts_ids))
                    self.predictor.release_feature(fi)
                pred_cnt += n_ready

    def _detect_objects(self, vid_size) -> Optional[BoxList]:
        if self.obj_det is None or not self._frames:
            return None
        kframe = self._frames[self._center]
        kdata  = self.obj_det.image_preprocess(kframe)
        im_dim = _torch.FloatTensor(
            (kframe.shape[1], kframe.shape[0])).repeat(1, 2)
        dets   = self.obj_det.images_detection(kdata, im_dim)
        if isinstance(dets, int) or dets.shape[0] == 0:
            obj_t = _torch.zeros((0, 4))
        else:
            obj_t = dets[:, 1:5].cpu()
        return BoxList(obj_t, vid_size, "xyxy").clip_to_image()

print("✅ AVAPredictor + AVAPredictorWorker  ready")

In [ ]:
from __future__ import annotations

import argparse
import os
import sys
import gc
import queue
import threading
from typing import Dict, List
from itertools import count as _count
from time import sleep as _sleep
from pathlib import Path

import torch as _torch
import torch.nn as _nn
import numpy as _np
import cv2 as _cv2
from tqdm import tqdm

_cv2.setNumThreads(0)

# Monkey patch for older PyTorch versions (from your original code) ---
if not getattr(_nn.Conv3d.__init__, "_patched", False):
    _orig_conv3d = _nn.Conv3d.__init__
    def _safe_conv3d(self, *args, **kwargs):
        def _s(x):
            if isinstance(x, (tuple, list)): return tuple(int(i) for i in x)
            try: return int(x)
            except (TypeError, ValueError): return x
        clean_args   = tuple(_s(a) if i in (2, 3, 4, 5) else a for i, a in enumerate(args))
        clean_kwargs = {k: (_s(v) if k in ("kernel_size", "stride", "padding", "dilation") else v)
                        for k, v in kwargs.items()}
        _orig_conv3d(self, *clean_args, **clean_kwargs)
    _safe_conv3d._patched = True
    _nn.Conv3d.__init__ = _safe_conv3d

# --- Helpers ---
def _video_info(path):
    cap  = _cv2.VideoCapture(path)
    info = dict(
        width    = int(cap.get(_cv2.CAP_PROP_FRAME_WIDTH)),
        height   = int(cap.get(_cv2.CAP_PROP_FRAME_HEIGHT)),
        fps      = cap.get(_cv2.CAP_PROP_FPS),
        n_frames = int(cap.get(_cv2.CAP_PROP_FRAME_COUNT)),
    )
    cap.release()
    return info

def _ms_to_timestamp(ms):
    ms = int(ms); msec = ms % 1000; ms //= 1000
    sec = ms % 60; ms //= 60; mins = ms % 60; hrs = ms // 60
    return f"{hrs:02d}:{mins:02d}:{sec:02d}.{msec:03d}"

def _boxes_to_list(boxes):
    if boxes is None: return []
    if isinstance(boxes, _torch.Tensor):
        return [tuple(float(v) for v in row) for row in boxes]
    return [tuple(float(v) for v in b) for b in boxes]

def _ids_to_list(ids):
    if ids is None: return []
    if isinstance(ids, _torch.Tensor):
        return [int(v) for v in ids.view(-1)]
    return [int(v) for v in ids]

# Optimized Visualizer
class AVAVisualizer:
    CATEGORIES: List[str] = [
        "bend/bow", "crawl", "crouch/kneel", "dance", "fall down",
        "get up", "jump/leap", "lie/sleep", "martial art", "run/jog",
        "sit", "stand", "swim", "walk",
        "answer phone", "brush teeth", "carry/hold sth.", "catch sth.",
        "chop", "climb", "clink glass", "close", "cook", "cut", "dig",
        "dress/put on clothing", "drink", "drive", "eat", "enter", "exit",
        "extract", "fishing", "hit sth.", "kick sth.", "lift/pick up",
        "listen to sth.", "open", "paint", "play board game",
        "play musical instrument", "play with pets", "point to sth.",
        "press", "pull sth.", "push sth.", "put down", "read", "ride",
        "row boat", "sail boat", "shoot", "shovel", "smoke", "stir",
        "take a photo", "look at a cellphone", "throw", "touch sth.",
        "turn", "watch screen", "work on a computer", "write",
        "fight/hit sb.", "give/serve sth. to sb.", "grab sb.", "hand clap",
        "hand shake", "hand wave", "hug sb.", "kick sb.", "kiss sb.",
        "lift sb.", "listen to sb.", "play with kids", "push sb.", "sing",
        "take sth. from sb.", "talk", "watch sb.",
    ]
    COMMON_CATES: List[str] = [
        "dance", "run/jog", "sit", "stand", "swim", "walk",
        "answer phone", "carry/hold sth.", "drive",
        "play musical instrument", "ride",
        "fight/hit sb.", "listen to sb.", "talk", "watch sb.",
    ]
    EXCLUSION: List[str] = [
        "crawl", "brush teeth", "catch sth.", "chop", "clink glass", "cook",
        "dig", "exit", "extract", "fishing", "kick sth.", "paint",
        "play board game", "play with pets", "press", "row boat", "shovel",
        "stir", "kick sb.", "play with kids",
    ]
    ACTION_COLORS = ((176, 85, 234), (87, 118, 198), (52, 189, 199)) # RGB

    def __init__(
        self,
        input_path,
        output_path,
        realtime,
        start,
        duration,
        show_time,
        confidence_threshold = 0.5,
        exclude_class        = None,
        common_cate          = False,
        enable_reid          = False, 
    ) -> None:
        self.info        = _video_info(input_path)
        self.realtime    = realtime
        self.start       = start
        self.duration    = duration
        self.show_time   = show_time
        self.thresh      = confidence_threshold
        self.enable_reid = enable_reid

        fps = self.info["fps"]
        if fps == 0 or fps > 100:
            print(f"Warning: suspicious FPS {fps}")

        if common_cate:
            self.cate_list = self.COMMON_CATES
            self.cat_split = (6, 11)
        else:
            self.cate_list = self.CATEGORIES
            self.cat_split = (14, 63)

        self.cls2id   = {n: i for i, n in enumerate(self.cate_list)}
        excl          = exclude_class if exclude_class is not None else self.EXCLUSION
        self.excl_ids = {self.cls2id[c] for c in excl if c in self.cls2id}

        W, H          = self.info["width"], self.info["height"]
        self.width    = W
        self.height   = H
        long_side     = min(W, H)
        
        # Optimized OpenCV Font Scaling
        self.box_width = max(int(round(long_side / 250)), 1)
        self.font = _cv2.FONT_HERSHEY_SIMPLEX
        self.font_scale = max(W, H) / 1500.0
        self.badge_font_scale = max(W, H) / 1800.0
        self.thickness = max(int(self.font_scale * 2), 1)

        self._action_dict: Dict = {}
        self._current_labels: Dict = {}
        self._valid_tracker_ids: set = set()

        self._reid = None
        # _reid is injected externally via make_visualizer(reid_cls=...).
        # AVAVisualizer never imports or instantiates a ReIdentifier itself.

        if realtime:
            fourcc       = _cv2.VideoWriter_fourcc(*"mp4v")
            self._outvid = _cv2.VideoWriter(output_path, fourcc, fps, (W, H))
        else:
            self._frame_q  = queue.Queue(maxsize=512)
            self._result_q = queue.Queue()
            self._track_q  = queue.Queue()
            self._done_q   = queue.Queue()

            self._loader = threading.Thread(
                target=self._load_frames, args=(input_path,),
                daemon=True, name="FrameLoader")
            self._writer = threading.Thread(
                target=self._write_frames, args=(output_path,),
                daemon=True, name="FrameWriter")
            self._loader.start()
            self._writer.start()

    @staticmethod
    def _box_area(box):
        x1, y1, x2, y2 = box
        return max(0.0, x2 - x1) * max(0.0, y2 - y1)

    def _filter_boxes(self, box_list, tracker_ids):
        frame_area = self.width * self.height
        min_area   = 0.015 * frame_area
        min_height = 0.20  * self.height
        min_width  = 0.030 * self.width
        paired = sorted(zip(box_list, tracker_ids),
                        key=lambda x: self._box_area(x[0]), reverse=True)
        kept_boxes, kept_ids = [], []
        for box, tid in paired:
            x1, y1, x2, y2 = box
            w = max(1.0, x2 - x1); h = max(1.0, y2 - y1)
            if self._box_area(box) < min_area: continue
            if h < min_height: continue
            if w < min_width:  continue
            if not (1.2 <= h / w <= 5.0): continue
            kept_boxes.append(box); kept_ids.append(tid)
        return kept_boxes, kept_ids

    def _run_reid(self, frame_bgr, boxes, ids) -> None:
        if not self.enable_reid:
            return
        tracker_ids = _ids_to_list(ids)
        box_list    = _boxes_to_list(boxes)
        if not tracker_ids:
            self._current_labels = {}; self._valid_tracker_ids = set(); return
        valid_boxes, valid_ids = self._filter_boxes(box_list, tracker_ids)
        self._valid_tracker_ids = set(valid_ids)
        if not valid_ids:
            self._current_labels = {}; return
        self._current_labels = self._reid.update(frame_bgr, valid_ids, valid_boxes)

    def _persistent_id_for(self, tracker_id: int) -> int:
        if self._reid is not None:
            pid = self._reid.get_persistent_id(tracker_id)
            if pid is not None:
                return pid
        return tracker_id

    def write_realtime_frame(self, result, orig_img, boxes, scores, ids):
        frame_bgr = orig_img[:, :, ::-1].copy()
        if boxes is not None:
            self._run_reid(frame_bgr, boxes, ids)
        if result is not None:
            pred, _ts, pred_ids = result
            self._update_action_dict(pred.get_field("scores"), pred_ids)
        if boxes is not None:
            # Optimized: Modifies frame_bgr directly
            frame_bgr = self._render_labels(frame_bgr, boxes, ids)
            
        _cv2.imshow("AlphAction", frame_bgr)
        self._outvid.write(frame_bgr)
        return _cv2.waitKey(1) != 27

    def send_result(self, result) -> None: self._result_q.put(result)
    def send_track(self, result) -> None:  self._track_q.put(result)

    def show_progress(self, total_frames) -> None:
        cnt = 0
        while True:
            try: self._done_q.get_nowait(); cnt += 1
            except queue.Empty: break
        pbar = tqdm(total=total_frames, initial=cnt,
                     desc="Video Writer", unit=" frame")
        while cnt < total_frames:
            self._done_q.get(); cnt += 1; pbar.update(1)
        pbar.close()

    def close(self) -> None:
        if self.realtime:
            self._outvid.release(); _cv2.destroyAllWindows()
        else:
            self._writer.join()
        if self._reid is not None:
            self._reid.reset()

    def _load_frames(self, path) -> None:
        cap = _cv2.VideoCapture(path)
        cap.set(_cv2.CAP_PROP_POS_MSEC, self.start)
        while True:
            ok, frame = cap.read()
            if not ok: break
            ms = cap.get(_cv2.CAP_PROP_POS_MSEC)
            if self.duration != -1 and ms > self.start + self.duration: break
            self._frame_q.put((frame, ms))
        cap.release()
        self._frame_q.put("DONE")

    def _write_frames(self, output_path) -> None:
        W, H   = self.info["width"], self.info["height"]
        fps    = self.info["fps"]
        fourcc = _cv2.VideoWriter_fourcc(*"mp4v")
        out    = _cv2.VideoWriter(output_path, fourcc, fps, (W, H))

        pred_result = self._result_q.get()
        pred_ts     = float("inf")
        pred_ids    = None
        if not isinstance(pred_result, str):
            pred_result, pred_ts, pred_ids = pred_result

        while True:
            track = self._track_q.get()
            data  = self._frame_q.get()
            if isinstance(pred_result, str) and data == "DONE":
                break
            frame, ms = data
            
            if self.show_time:
                frame = self._draw_timestamp(frame, ms)
                
            if ms - pred_ts + 0.5 > 0:
                boxes  = pred_result.bbox
                scores = pred_result.get_field("scores")
                ids    = pred_ids
                pred_result = self._result_q.get()
                if not isinstance(pred_result, str):
                    pred_result, pred_ts, pred_ids = pred_result
                else:
                    pred_ts = float("inf")
            else:
                boxes, ids = track
                scores     = None
                
            if boxes is not None:
                self._run_reid(frame, boxes, ids)
                self._update_action_dict(scores, ids)
                # Optimized: Blends and draws directly onto the frame 
                frame = self._render_labels(frame, boxes, ids)
                
            out.write(frame)
            self._done_q.put(True)

        out.release()
        tqdm.write("Output video written.")

    def _update_action_dict(self, scores, ids) -> None:
        if scores is None: return
        for score, tid_val in zip(scores, ids):
            tid = int(tid_val) if not isinstance(tid_val, int) else tid_val
            if self.enable_reid and self._valid_tracker_ids and tid not in self._valid_tracker_ids:
                continue
            pid = self._persistent_id_for(tid)
            above = _torch.nonzero(score >= self.thresh, as_tuple=False).squeeze(1).tolist()
            captions, colors = [], []
            for cid in above:
                if cid in self.excl_ids: continue
                captions.append(f"{self.cate_list[cid]} {score[cid]:.2f}")
                colors.append(
                    0 if cid < self.cat_split[0] else
                    1 if cid < self.cat_split[1] else 2
                )
            self._action_dict[pid] = {"captions": captions, "colors": colors}

    def _render_labels(self, frame_bgr, boxes, ids):
        overlay = frame_bgr.copy()
        tracker_ids = _ids_to_list(ids)
        box_list    = _boxes_to_list(boxes)

        # 1. Draw backgrounds on overlay for transparency
        for box, tid in zip(box_list, tracker_ids):
            if self.enable_reid and self._valid_tracker_ids and tid not in self._valid_tracker_ids:
                continue

            x1, y1, x2, y2 = [int(round(v)) for v in box]
            
            if self.enable_reid:
                pid       = self._persistent_id_for(tid)
                label_str = self._current_labels.get(tid, f"Person {pid}")
                color_bgr = person_color_rgb(pid)[::-1] # RGB to BGR
                
                _cv2.rectangle(overlay, (x1, y1), (x2, y2), color_bgr, self.box_width)
                (tw, th), baseline = _cv2.getTextSize(label_str, self.font, self.badge_font_scale, self.thickness)
                pad = 4
                _cv2.rectangle(overlay, (x1, y1), (x1 + tw + pad*2, y1 + th + pad*2 + baseline), color_bgr, -1)
            else:
                _cv2.rectangle(overlay, (x1, y1), (x2, y2), (40, 40, 110), self.box_width)

            pid  = self._persistent_id_for(tid)
            info = self._action_dict.get(pid)
            if not info or not info["captions"]: continue

            max_w = 0; total_h = 0; text_sizes = []
            pad = 6; gap = 4
            
            for caption in info["captions"]:
                (tw, th), baseline = _cv2.getTextSize(caption, self.font, self.font_scale, self.thickness)
                text_sizes.append(((tw, th), baseline))
                max_w = max(max_w, tw)
                total_h += (th + pad * 2 + gap)

            start_y = max(y1 - total_h, gap)

            for i, caption in enumerate(info["captions"]):
                rx1 = x1
                ry1 = start_y + int((text_sizes[0][0][1] + pad*2 + gap) * i)
                (tw, th), baseline = text_sizes[i]
                color_bgr = self.ACTION_COLORS[info["colors"][i]][::-1] # RGB to BGR
                _cv2.rectangle(overlay, (rx1, ry1), (rx1 + tw + pad*2, ry1 + th + pad*2), color_bgr, -1)

        # 2. Apply transparency blend
        alpha = 0.65
        _cv2.addWeighted(overlay, alpha, frame_bgr, 1 - alpha, 0, frame_bgr)

        # 3. Draw solid text on top
        for box, tid in zip(box_list, tracker_ids):
            if self.enable_reid and self._valid_tracker_ids and tid not in self._valid_tracker_ids:
                continue
                
            x1, y1, x2, y2 = [int(round(v)) for v in box]
            
            if self.enable_reid:
                pid       = self._persistent_id_for(tid)
                label_str = self._current_labels.get(tid, f"Person {pid}")
                (tw, th), baseline = _cv2.getTextSize(label_str, self.font, self.badge_font_scale, self.thickness)
                _cv2.putText(frame_bgr, label_str, (x1 + 4, y1 + th + 4), self.font, self.badge_font_scale, (255, 255, 255), self.thickness, _cv2.LINE_AA)
                
            pid  = self._persistent_id_for(tid)
            info = self._action_dict.get(pid)
            if not info or not info["captions"]: continue

            total_h = 0
            for caption in info["captions"]:
                (tw, th), baseline = _cv2.getTextSize(caption, self.font, self.font_scale, self.thickness)
                total_h += (th + 12 + 4) 
            start_y = max(y1 - total_h, 4)

            for i, caption in enumerate(info["captions"]):
                (tw, th), baseline = _cv2.getTextSize(caption, self.font, self.font_scale, self.thickness)
                rx1 = x1
                ry1 = start_y + int((th + 12 + 4) * i)
                _cv2.putText(frame_bgr, caption, (rx1 + 6, ry1 + th + 6), self.font, self.font_scale, (255, 255, 255), self.thickness, _cv2.LINE_AA)

        return frame_bgr

    def _draw_timestamp(self, frame, ms):
        text = _ms_to_timestamp(ms)
        (tw, th), baseline = _cv2.getTextSize(text, self.font, self.font_scale, self.thickness)
        pad = 8
        y0 = self.height - th - pad * 2
        
        overlay = frame.copy()
        _cv2.rectangle(overlay, (0, y0), (tw + pad * 2, self.height), (0, 0, 0), -1)
        _cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
        
        _cv2.putText(frame, text, (pad, y0 + th + pad), self.font, self.font_scale, (255, 255, 255), self.thickness, _cv2.LINE_AA)
        return frame

print("✅ AVAVisualizer Ready")

In [ ]:
from __future__ import annotations

import argparse
import os
import sys
import gc
import random
from itertools import count as _count
from time import sleep as _sleep
from pathlib import Path

import torch as _torch
import torch.nn as _nn
import numpy as _np
import cv2 as _cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display as _display
from tqdm import tqdm

if not getattr(_nn.Conv3d.__init__, "_patched", False):
    _orig_conv3d = _nn.Conv3d.__init__
    def _safe_conv3d(self, *args, **kwargs):
        def _s(x):
            if isinstance(x, (tuple, list)): return tuple(int(i) for i in x)
            try: return int(x)
            except (TypeError, ValueError): return x
        clean_args   = tuple(_s(a) if i in (2, 3, 4, 5) else a for i, a in enumerate(args))
        clean_kwargs = {k: (_s(v) if k in ("kernel_size", "stride", "padding", "dilation") else v)
                        for k, v in kwargs.items()}
        _orig_conv3d(self, *clean_args, **clean_kwargs)
    _safe_conv3d._patched = True
    _nn.Conv3d.__init__ = _safe_conv3d


class BaseDemo:
    def __init__(
        self,
        video_path:        str   = "test_video.mp4",
        output_path:       str   = "output.mp4",
        enable_reid:       bool  = False,
        cpu:               bool  = False,
        visual_threshold:  float = 0.5,
        start:             int   = 0,
        duration:          int   = -1,
        detect_rate:       int   = 4,
        common_cate:       bool  = False,
        hide_time:         bool  = False,
        tracker_box_thres: float = 0.1,
        tracker_nms_thres: float = 0.4,
        reid_weights:      str   = None,
    ) -> None:
        self.args = argparse.Namespace(
            video_path        = video_path,
            output_path       = output_path,
            input_path        = video_path,
            cfg_path          = str(MODEL_CFG),
            weight_path       = str(MODEL_WEIGHTS),
            device            = _torch.device("cpu" if cpu else "cuda"),
            realtime          = False,
            webcam            = False,
            gpus              = [0] if _torch.cuda.device_count() >= 1 else [-1],
            min_box_area      = 0,
            tracking          = True,
            detector          = "tracker",
            debug             = False,
            start             = start,
            duration          = duration,
            detect_rate       = detect_rate,
            visual_threshold  = visual_threshold,
            common_cate       = common_cate,
            hide_time         = hide_time,
            tracker_box_thres = tracker_box_thres,
            tracker_nms_thres = tracker_nms_thres,
        )
        self.enable_reid      = enable_reid
        self.reid_weights     = reid_weights

    def make_visualizer(self, reid_cls=None, **reid_kwargs) -> AVAVisualizer:
        vw = AVAVisualizer(
            input_path           = self.args.input_path,
            output_path          = self.args.output_path,
            realtime             = False,
            start                = self.args.start,
            duration             = self.args.duration,
            show_time            = not self.args.hide_time,
            confidence_threshold = self.args.visual_threshold,
            common_cate          = self.args.common_cate,
            enable_reid          = self.enable_reid,
        )
        if self.enable_reid and reid_cls is not None:
            import inspect as _inspect
            default_kwargs = dict(
                use_deep_features = True,
                use_bg_crop       = True,
                use_quality_ema   = True,
            )
            if self.reid_weights:
                default_kwargs["weight_path"] = self.reid_weights
            _accepted = set(_inspect.signature(reid_cls).parameters)
            _kwargs   = {k: v for k, v in {**default_kwargs, **reid_kwargs}.items()
                         if k in _accepted}
            vw._reid = reid_cls(**_kwargs)
        return vw

    def make_predictor(self) -> AVAPredictorWorker:
        return AVAPredictorWorker(self.args)

    def on_frame(self, frame_idx: int) -> None:
        pass

    def run(self, reid_cls=None, **reid_kwargs) -> None:
        try:
            from alphaction.config import cfg as _acfg
            _acfg.defrost()
        except Exception:
            pass

        print(f"  Input   : {self.args.input_path}")
        print(f"  Output  : {self.args.output_path}")
        print(f"  Device  : {self.args.device}")
        print(f"  Re-ID   : {'enabled' if self.enable_reid else 'disabled (baseline)'}")

        video_writer = self.make_visualizer(reid_cls=reid_cls, **reid_kwargs)
        predictor    = self.make_predictor()
        pred_done    = False
        frame_idx    = 0
        success      = False

        print("\n🚀 Tracking frames (action prediction runs in background threads)…")
        try:
            for frame_idx in tqdm(_count(), desc="Tracker", unit=" frame"):
                if not predictor._thread.is_alive() and not pred_done:
                    print("\n❌ Background prediction thread crashed — check traceback above.")
                    break

                with _torch.no_grad():
                    orig_img, boxes, scores, ids = predictor.read_track()

                if orig_img is None:
                    predictor.signal_tracking_done()
                    break

                self.on_frame(frame_idx)
                video_writer.send_track((boxes, ids))
                while not pred_done:
                    result = predictor.read_result()
                    if result is None:
                        break
                    elif result == "done":
                        pred_done = True
                    else:
                        video_writer.send_result(result)

            if predictor._thread.is_alive():
                video_writer.send_track("DONE")
                while not pred_done:
                    result = predictor.read_result()
                    if result is None:
                        _sleep(0.05)
                    elif result == "done":
                        pred_done = True
                    else:
                        video_writer.send_result(result)
                video_writer.send_result("DONE")
                video_writer.show_progress(frame_idx)
                success = True
                print(f"\n✅ Done. Output saved to: {self.args.output_path}")

        except KeyboardInterrupt:
            print("\n⚠️ Interrupted by user.")
        except Exception as e:
            import traceback
            print(f"\n❌ Pipeline error: {e}")
            traceback.print_exc()

        finally:
            print("🧹 Cleaning up…")
            try: video_writer.close()
            except Exception: pass
            try: predictor.terminate()
            except Exception: pass
            gc.collect()
            if _torch.cuda.is_available():
                _torch.cuda.empty_cache()
            print("✅ Cleanup complete.")

        if success:
            print("\n🖼️  Generating frame preview…")
            subtitle = "Baseline" if not self.enable_reid else "With Re-ID"
            try:
                NotebookUtils.compare_videos(
                    videos   = [
                        {"path": self.args.input_path,  "label": "BEFORE  Re-ID  (raw video)"},
                        {"path": self.args.output_path, "label": f"Annotated Output  ({subtitle})"},
                    ],
                    n_frames = 5,
                    title    = Path(self.args.input_path).stem,
                )
            except NameError:
                pass # NotebookUtils is not defined in this snippet

print("✅ BaseDemo ready")
print("   Frame previews: NotebookUtils.compare_videos([...], n_frames=5)")

# 👉 Run demo.py

In [ ]:
os.chdir(str(DEMO_DIR))

demo = BaseDemo(
    video_path       = "test_video.mp4",
    output_path      = "output_baseline.mp4",
    enable_reid      = False,
    visual_threshold = 0.5,
    detect_rate      = 4,
)

demo.run()

# 👉 View Video Ouput

In [ ]:
NotebookUtils.show_video("output_baseline.mp4")
NotebookUtils.cleanup_gpu() # flush the VRAM for the next cell

# 👉 Applying Our Person Re-ID
Person Re-Identification (Re-ID) solves the problem of assigning a **stable, persistent identity** to each
person across an entire video — even when they temporarily leave the frame or the tracker resets their ID.

We evaluate **four progressively more powerful methods**


In [ ]:
from __future__ import annotations

import random
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

import cv2
import numpy as np

@dataclass
class TrackSegment:
    track_id:  int
    frame_ids: List[int]                            = field(default_factory=list)
    boxes:     List[Tuple[float,float,float,float]] = field(default_factory=list)
    def __len__(self): return len(self.frame_ids)

@dataclass
class EvalResults:
    config_name:        str
    precision:          float = 0.0
    recall:             float = 0.0
    f1:                 float = 0.0
    id_switches:        int   = 0
    total_frames:       int   = 0
    id_switches_per100: float = 0.0
    consistency:        float = 0.0
    total_raw_ids:      int   = 0
    total_persistent:   int   = 0
    id_reduction_pct:   float = 0.0
    ids_per_frame_raw:  float = 0.0
    ids_per_frame_reid: float = 0.0
    frag_rate:          float = 0.0
    n_reentry_events:   int   = 0
    n_tp:               int   = 0
    n_fp:               int   = 0
    n_fn:               int   = 0
    frame_snapshots:    List[Tuple[int,int,int]] = field(default_factory=list)

    def summary_dict(self) -> dict:
        return {
            "Config":           self.config_name,
            "Precision":        round(self.precision, 4),
            "Recall":           round(self.recall, 4),
            "F1":               round(self.f1, 4),
            "ID-Sw/100fr":      round(self.id_switches_per100, 2),
            "Consistency":      round(self.consistency, 4),
            "IDs/fr (raw)":     round(self.ids_per_frame_raw, 2),
            "IDs/fr (ReID)":    round(self.ids_per_frame_reid, 2),
            "Total raw IDs":    self.total_raw_ids,
            "Total persistent": self.total_persistent,
            "ID reduction %":   round(self.id_reduction_pct, 1),
            "Frag. rate":       round(self.frag_rate, 3),
        }

    def __str__(self):
        return (
            f"[{self.config_name}]\n"
            f"  Precision={self.precision:.4f}  Recall={self.recall:.4f}  F1={self.f1:.4f}\n"
            f"  ID-Sw/100fr={self.id_switches_per100:.2f}  Consistency={self.consistency:.4f}\n"
            f"  Re-entry events={self.n_reentry_events}  TP={self.n_tp}  FP={self.n_fp}  FN={self.n_fn}\n"
            f"  Raw IDs={self.total_raw_ids}  Persistent={self.total_persistent}  "
            f"Reduction={self.id_reduction_pct:.1f}%  Frag={self.frag_rate:.3f}"
        )

def _iou(a, b) -> float:
    ax1,ay1,ax2,ay2 = a;  bx1,by1,bx2,by2 = b
    ix1=max(ax1,bx1); iy1=max(ay1,by1)
    ix2=min(ax2,bx2); iy2=min(ay2,by2)
    inter = max(0.0,ix2-ix1)*max(0.0,iy2-iy1)
    ua    = (ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
    return inter/ua if ua>0 else 0.0

def _merge_boxes(boxes, thresh=0.25):
    if not boxes: return []
    boxes = sorted(boxes, key=lambda b:(b[2]-b[0])*(b[3]-b[1]), reverse=True)
    used  = [False]*len(boxes)
    out   = []
    for i,b in enumerate(boxes):
        if used[i]: continue
        grp = [b]
        for j in range(i+1,len(boxes)):
            if not used[j] and _iou(b,boxes[j])>thresh:
                grp.append(boxes[j]); used[j]=True
        out.append((min(g[0] for g in grp), min(g[1] for g in grp),
                    max(g[2] for g in grp), max(g[3] for g in grp)))
    return out

def extract_tracks_from_video(
    video_path: str,
    min_track_frames: int = 25,
    max_frames: int = 400,
) -> Tuple[List[np.ndarray], List[TrackSegment]]:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open: {video_path}")

    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fgbg     = cv2.createBackgroundSubtractorMOG2(200, 50, False)
    min_area = 0.004 * W * H
    min_h    = 0.10  * H
    max_age  = 8

    active: Dict[int,dict]         = {}
    store:  Dict[int,TrackSegment] = {}
    next_tid = 1
    frames: List[np.ndarray] = []

    for frame_idx in range(max_frames):
        ok, frame = cap.read()
        if not ok: break
        frames.append(frame.copy())

        fg = fgbg.apply(frame)
        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN,
             cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)))
        fg = cv2.dilate(fg, np.ones((15,15),np.uint8), iterations=2)

        cnts,_ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cands  = []
        for c in cnts:
            x,y,w,h = cv2.boundingRect(c)
            if w*h < min_area: continue
            if h   < min_h:    continue
            if h   < w:        continue
            cands.append((float(x),float(y),float(x+w),float(y+h)))
        cands = _merge_boxes(cands)

        used_c = set()
        for tid, ts in active.items():
            best_iou, best_ci = 0.0, -1
            for ci,cb in enumerate(cands):
                if ci in used_c: continue
                v = _iou(ts["box"], cb)
                if v > best_iou: best_iou,best_ci = v,ci
            if best_iou > 0.15 and best_ci >= 0:
                cb = cands[best_ci]
                ts["box"] = cb; ts["last"] = frame_idx
                store[tid].frame_ids.append(frame_idx)
                store[tid].boxes.append(cb)
                used_c.add(best_ci)

        for ci,cb in enumerate(cands):
            if ci not in used_c:
                tid = next_tid; next_tid += 1
                active[tid] = {"box":cb,"last":frame_idx}
                store[tid]  = TrackSegment(track_id=tid)
                store[tid].frame_ids.append(frame_idx)
                store[tid].boxes.append(cb)

        stale = [t for t,ts in active.items() if frame_idx-ts["last"] > max_age]
        for t in stale: del active[t]

    cap.release()
    valid = [t for t in store.values() if len(t) >= min_track_frames]
    return frames, valid

def evaluate_config(
    frames:      List[np.ndarray],
    tracks:      List[TrackSegment],
    reid,
    config_name: str,
    seed:        int = 42,
) -> EvalResults:
    rng = random.Random(seed)
    reid.reset()

    if not tracks:
        return EvalResults(config_name=config_name)

    max_existing  = max(t.track_id for t in tracks)
    new_id_start  = max_existing + 100

    split_info: Dict[int, Tuple[int,int]] = {}
    counter = 0
    for t in tracks:
        if len(t) < 4:
            continue
        lo  = max(1, len(t)//3)
        hi  = min(len(t)-1, 2*len(t)//3)
        mid = rng.randint(lo, hi)
        split_info[t.track_id] = (mid, new_id_start + counter)
        counter += 1

    frame_lookup: Dict[int, List] = {}
    for t in tracks:
        if t.track_id not in split_info:
            mid, new_tid = len(t), -1
        else:
            mid, new_tid = split_info[t.track_id]
        for i, (fid, box) in enumerate(zip(t.frame_ids, t.boxes)):
            if fid >= len(frames):
                continue
            is_second  = (i >= mid) and (new_tid != -1)
            assigned   = new_tid if is_second else t.track_id
            frame_lookup.setdefault(fid, []).append(
                (assigned, box, t.track_id, is_second)
            )

    first_half_pids:  Dict[int, List[int]] = {}
    second_half_pids: Dict[int, List[int]] = {}
    snapshots: List[Tuple[int,int,int]] = []
    prev_pid: Dict[int,int] = {}
    id_switches = 0
    raw_ids_per_frame:  List[int] = []
    reid_ids_per_frame: List[int] = []
    all_raw_tids:  set = set()
    all_reid_pids: set = set()

    for fid in sorted(frame_lookup.keys()):
        entries   = frame_lookup[fid]
        bgr       = frames[fid]
        tids      = [e[0] for e in entries]
        boxes     = [e[1] for e in entries]
        orig_tids = [e[2] for e in entries]
        is_second = [e[3] for e in entries]

        all_raw_tids.update(tids)
        raw_ids_per_frame.append(len(set(tids)))

        reid.update(bgr, tids, boxes)

        frame_pids: set = set()
        for tid, orig_tid, second in zip(tids, orig_tids, is_second):
            pid = reid.get_persistent_id(tid)
            if pid is None:
                continue
            frame_pids.add(pid)
            all_reid_pids.add(pid)

            if second:
                second_half_pids.setdefault(orig_tid, []).append(pid)
            else:
                first_half_pids.setdefault(orig_tid, []).append(pid)
                if tid in prev_pid and prev_pid[tid] != pid:
                    id_switches += 1
                prev_pid[tid] = pid

        reid_ids_per_frame.append(len(frame_pids))
        snapshots.append((fid, len(set(tids)), len(frame_pids)))

    tp = fp = fn = 0
    reentry_events = 0
    for orig_tid in split_info:
        reentry_events += 1
        fh = first_half_pids.get(orig_tid, [])
        sh = second_half_pids.get(orig_tid, [])
        if not fh or not sh:
            fn += 1
            continue
        dom_fh = max(set(fh), key=fh.count)
        dom_sh = max(set(sh), key=sh.count)
        if dom_fh == dom_sh:
            tp += 1
        else:
            fp += 1
    fn = max(0, reentry_events - tp - fp)

    precision = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    recall    = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    f1        = (2*precision*recall/(precision+recall)
                 if (precision+recall) > 0 else 0.0)

    total_fr  = len(snapshots)
    sw_per100 = id_switches/total_fr*100 if total_fr > 0 else 0.0

    consistencies = []
    for orig_tid in split_info:
        pids = first_half_pids.get(orig_tid, [])
        if not pids: continue
        dom = max(set(pids), key=pids.count)
        consistencies.append(pids.count(dom)/len(pids))
    consistency = float(np.mean(consistencies)) if consistencies else 0.0

    total_raw  = len(all_raw_tids)
    total_reid = len(all_reid_pids)
    id_red_pct = (total_raw-total_reid)/total_raw*100 if total_raw > 0 else 0.0
    frag_rate  = total_raw/total_reid if total_reid > 0 else 1.0

    return EvalResults(
        config_name        = config_name,
        precision          = precision,
        recall             = recall,
        f1                 = f1,
        id_switches        = id_switches,
        total_frames       = total_fr,
        id_switches_per100 = sw_per100,
        consistency        = consistency,
        total_raw_ids      = total_raw,
        total_persistent   = total_reid,
        id_reduction_pct   = id_red_pct,
        ids_per_frame_raw  = float(np.mean(raw_ids_per_frame))  if raw_ids_per_frame  else 0.0,
        ids_per_frame_reid = float(np.mean(reid_ids_per_frame)) if reid_ids_per_frame else 0.0,
        frag_rate          = frag_rate,
        n_reentry_events   = reentry_events,
        n_tp               = tp,
        n_fp               = fp,
        n_fn               = fn,
        frame_snapshots    = snapshots,
    )

def print_before_after_summary(result, config_name):
    snaps = result.frame_snapshots
    if not snaps:
        return
    import pandas as _pd2
    from IPython.display import display as _d2, HTML as _H2
    raw_vals  = [s[1] for s in snaps]
    reid_vals = [s[2] for s in snaps]
    step = max(1, len(snaps) // 20)
    rows = [{"Frame": s[0], "Before Re-ID (raw IDs)": s[1], "After Re-ID (persistent IDs)": s[2]}
            for s in snaps[::step][:20]]
    agg = {
        "Metric":  ["Mean persons/frame", "Max persons/frame", "Frames ≥ 2 persons",
                    "Total unique IDs", "ID reduction", "Re-entry events", "TP (re-ID correct)",
                    "Precision", "Recall", "F1"],
        "Before":  [f"{np.mean(raw_vals):.2f}", f"{max(raw_vals)}", f"{sum(1 for v in raw_vals if v>=2)}",
                    str(result.total_raw_ids), "—", str(result.n_reentry_events), "—", "—", "—", "—"],
        "After":   [f"{np.mean(reid_vals):.2f}", f"{max(reid_vals)}", f"{sum(1 for v in reid_vals if v>=2)}",
                    str(result.total_persistent), f"{result.id_reduction_pct:.1f}%", "—",
                    str(result.n_tp), f"{result.precision:.4f}", f"{result.recall:.4f}", f"{result.f1:.4f}"],
    }
    _d2(_H2(f"<h4>Before vs After Re-ID — [{config_name}]</h4>"))
    try:
        _d2(_pd2.DataFrame(rows).style.hide(axis="index").set_properties(**{"text-align":"center"}))
        _d2(_pd2.DataFrame(agg).style.hide(axis="index").set_properties(**{"text-align":"center"}))
    except AttributeError:
        _d2(_pd2.DataFrame(rows).style.hide_index().set_properties(**{"text-align":"center"}))
        _d2(_pd2.DataFrame(agg).style.hide_index().set_properties(**{"text-align":"center"}))

import pandas as _pd
import plotly.graph_objects as _go
from plotly.subplots import make_subplots as _make_subplots
from IPython.display import display as _display, HTML as _HTML

_METHOD_COLORS = {
    "HSV Baseline":      "#4C78A8",
    "ResNet50 Base":     "#F58518",
    "ResNet50 Finetune": "#E45756",
    "InsightFace":       "#54A24B",
}

print("✅ reid_evaluator  +  show_reid_results  ready")

import pandas as _pd
import plotly.graph_objects as _go
from plotly.subplots import make_subplots as _make_subplots
from IPython.display import display as _display, HTML as _HTML

_METHOD_COLORS = {
    "HSV Baseline":      "#4C78A8",
    "ResNet50 Base":     "#F58518",
    "ResNet50 Finetune": "#E45756",
    "InsightFace":       "#54A24B",
}

def _style_table(df):
    styler = df.style.set_properties(**{"text-align": "center"}).set_table_styles([
        {"selector": "th", "props": [("background-color", "#1e1e3f"), ("color", "white"),
                                     ("font-weight", "bold"), ("text-align", "center"), ("padding", "8px")]},
        {"selector": "tr:nth-child(even)", "props": [("background-color", "#f0f4ff")]},
        {"selector": "td", "props": [("padding", "6px 12px")]},
    ])
    try:
        return styler.hide(axis="index")
    except AttributeError:
        return styler.hide_index()

def show_reid_results(results_list, title="Re-ID Evaluation Results"):
    rows = []
    for r in results_list:
        rows.append({
            "Method":         r.config_name,
            "Precision":      f"{r.precision:.4f}",
            "Recall":         f"{r.recall:.4f}",
            "F1":             f"{r.f1:.4f}",
            "ID-Sw/100f":     f"{r.id_switches_per100:.2f}",
            "Consistency":    f"{r.consistency:.4f}",
            "ID Reduction %": f"{r.id_reduction_pct:.1f}%",
            "Frag. Rate":     f"{r.frag_rate:.3f}",
        })
    df = _pd.DataFrame(rows)
    _display(_HTML(f"<h4>📊 {title}</h4>"))
    _display(_style_table(df))

    if len(results_list) > 1:
        names  = [r.config_name for r in results_list]
        colors = [_METHOD_COLORS.get(n, "#888") for n in names]
        fig = _make_subplots(
            rows=2, cols=3,
            subplot_titles=("F1 Score", "Precision", "Recall",
                            "ID-Sw/100f (lower=better)", "Consistency", "ID Reduction %"),
            vertical_spacing=0.22, horizontal_spacing=0.10,
        )
        def bar(vals, row, col, pct=False):
            fmt = ["{:.1%}".format(v) if pct else "{:.4f}".format(v) for v in vals]
            fig.add_trace(_go.Bar(x=names, y=vals, marker_color=colors,
                                  text=fmt, textposition="outside", showlegend=False),
                          row=row, col=col)
        bar([r.f1                      for r in results_list], 1, 1)
        bar([r.precision               for r in results_list], 1, 2)
        bar([r.recall                  for r in results_list], 1, 3)
        bar([r.id_switches_per100      for r in results_list], 2, 1)
        bar([r.consistency             for r in results_list], 2, 2)
        bar([r.id_reduction_pct / 100  for r in results_list], 2, 3, pct=True)
        fig.update_layout(
            title_text=f"<b>{title} — Visual Comparison</b>",
            height=540, width=1000,
            paper_bgcolor="white", plot_bgcolor="#f9f9f9",
            margin=dict(t=90, b=40, l=50, r=30),
        )
        fig.show(renderer="iframe")

print("✅ reid_evaluator  +  show_reid_results  ready")

# ❄️ METHOD 1 - HSV Colour Histogram Re-ID

**How it works:** Extracts an L2-normalised 2D HSV histogram from each person crop,
divided into three overlapping spatial strips (head, torso, legs) with different weights.
Matching uses weighted cosine distance — no neural network involved.

**Strengths:** Fast, runs on CPU, no pretrained weights needed.  
**Weakness:** Sensitive to lighting changes and similar clothing across people.

**Key ablation flags:** `use_bg_crop`, `use_strips`, `use_occlusion`, `use_quality_ema`


In [ ]:
from __future__ import annotations
from typing import Dict, List, Optional, Tuple
import cv2
import numpy as np

PERSON_COLORS_RGB: Tuple[Tuple[int, int, int], ...] = (
    ( 46, 204, 113),   # Person  1 — green
    ( 52, 152, 219),   # Person  2 — blue
    (155,  89, 182),   # Person  3 — purple
    (241, 196,  15),   # Person  4 — yellow
    ( 26, 188, 156),   # Person  5 — teal
    (230, 126,  34),   # Person  6 — orange
    (231,  76,  60),   # Person  7 — red
    ( 52,  73,  94),   # Person  8 — dark slate
    (149, 165, 166),   # Person  9 — silver
    (211,  84,   0),   # Person 10 — burnt orange
    ( 39, 174,  96),   # Person 11 — dark green
    (142,  68, 173),   # Person 12 — dark purple
)

def person_color_rgb(persistent_id: int) -> Tuple[int, int, int]:
    return PERSON_COLORS_RGB[(persistent_id - 1) % len(PERSON_COLORS_RGB)]

_STRIPS: Tuple[Tuple[float, float, float], ...] = (
    (0.00, 0.40, 0.20),  # Strip 0: Head / shoulders — weight 0.20
    (0.25, 0.75, 0.45),  # Strip 1: Torso           — weight 0.45 (dominant)
    (0.60, 1.00, 0.35),  # Strip 2: Legs / feet     — weight 0.35
)

_MIN_STRIP_PX: int   = 20    # minimum strip height to be included
_BG_MARGIN:   float  = 0.10  # fraction of box width to crop each side

_MIN_UPDATE_QUALITY: float = 0.008   # ~0.8% of frame area

_REASSIGNMENT_DIST: float = 0.55

class HSVPersonReIdentifier:
    def __init__(
        self,
        reid_threshold:        float = 0.22,
        reid_threshold_relax:  float = 0.38,
        ema_alpha_near:        float = 0.80,
        ema_alpha_far:         float = 0.95,
        max_gallery_size:      int   = 64,
        h_bins:                int   = 32,
        s_bins:                int   = 32,
        use_bg_crop:     bool = True,
        use_strips:      bool = True,
        use_occlusion:   bool = True,
        use_quality_ema: bool = True,
    ) -> None:
        self.threshold        = reid_threshold
        self.threshold_relax  = reid_threshold_relax
        self.alpha_near       = ema_alpha_near
        self.alpha_far        = ema_alpha_far
        self.max_gallery_size = max_gallery_size
        self.h_bins           = h_bins
        self.s_bins           = s_bins

        self.use_bg_crop     = use_bg_crop
        self.use_strips      = use_strips
        self.use_occlusion   = use_occlusion
        self.use_quality_ema = use_quality_ema

        self._gallery:               Dict[int, Dict] = {}
        self._tracker_to_persistent: Dict[int, int]  = {}
        self._next_pid:  int = 1
        self._frame_idx: int = 0

    def _strip_histogram(self, hsv_crop: np.ndarray) -> np.ndarray:
        if hsv_crop.size == 0:
            return np.zeros(self.h_bins * self.s_bins, dtype=np.float32)
        hist = cv2.calcHist(
            [hsv_crop], [0, 1], None,
            [self.h_bins, self.s_bins],
            [0, 180, 0, 256],
        ).flatten().astype(np.float32)
        norm = np.linalg.norm(hist)
        return hist / (norm + 1e-8)

    def _extract_strips(
        self,
        frame_bgr: np.ndarray,
        box:       Tuple[float, float, float, float],
    ) -> Tuple[List[Optional[np.ndarray]], float]:
        x1 = max(0, int(box[0]))
        y1 = max(0, int(box[1]))
        x2 = min(frame_bgr.shape[1], int(box[2]))
        y2 = min(frame_bgr.shape[0], int(box[3]))

        bw = max(1, x2 - x1)
        bh = max(1, y2 - y1)

        if self.use_bg_crop:
            margin = int(bw * _BG_MARGIN)
            cx1 = x1 + margin
            cx2 = x2 - margin
            if cx2 <= cx1 + 4:
                cx1, cx2 = x1, x2   # box too narrow — skip crop
        else:
            cx1, cx2 = x1, x2

        full_crop = frame_bgr[y1:y2, cx1:cx2]
        ch = full_crop.shape[0]

        if ch == 0 or full_crop.shape[1] == 0:
            return [None, None, None], 0.0

        hsv_crop = cv2.cvtColor(full_crop, cv2.COLOR_BGR2HSV)

        if self.use_strips:
            strips: List[Optional[np.ndarray]] = []
            for r_start, r_end, _w in _STRIPS:
                row0 = int(ch * r_start)
                row1 = int(ch * r_end)
                crop = hsv_crop[row0:row1, :]
                if self.use_occlusion and crop.shape[0] < _MIN_STRIP_PX:
                    strips.append(None)
                else:
                    strips.append(self._strip_histogram(crop))
        else:
            global_hist = self._strip_histogram(hsv_crop)
            strips = [global_hist, global_hist, global_hist]

        frame_area = max(1, frame_bgr.shape[0] * frame_bgr.shape[1])
        quality    = (bw * bh) / frame_area
        return strips, quality

    @staticmethod
    def _cosine_dist(a: np.ndarray, b: np.ndarray) -> float:
        return float(1.0 - np.dot(a, b))

    def _weighted_distance(
        self,
        query_strips:   List[Optional[np.ndarray]],
        gallery_strips: List[Optional[np.ndarray]],
    ) -> Optional[float]:
        active_w = 0.0
        dist_acc = 0.0
        for i, (_r0, _r1, w) in enumerate(_STRIPS):
            q = query_strips[i]
            g = gallery_strips[i]
            if q is None or g is None:
                continue
            dist_acc += w * self._cosine_dist(q, g)
            active_w += w
        if active_w < 1e-6:
            return None
        return dist_acc / active_w

    def _register_new_person(self, strips: List[Optional[np.ndarray]]) -> int:
        if len(self._gallery) >= self.max_gallery_size:
            oldest = min(self._gallery,
                         key=lambda k: self._gallery[k]["last_seen"])
            del self._gallery[oldest]
        pid = self._next_pid
        self._next_pid += 1
        self._gallery[pid] = {
            "strips":    strips,
            "last_seen": self._frame_idx,
            "n_obs":     1,
        }
        return pid

    def _update_gallery(
        self,
        pid:     int,
        strips:  List[Optional[np.ndarray]],
        quality: float,
    ) -> None:
        if self.use_quality_ema:
            alpha = self.alpha_near if quality >= 0.01 else self.alpha_far
        else:
            alpha = self.alpha_near

        stored     = self._gallery[pid]["strips"]
        new_strips: List[Optional[np.ndarray]] = []

        for i in range(len(_STRIPS)):
            s_new = strips[i]
            s_old = stored[i]
            if s_new is None:
                new_strips.append(s_old)
            elif s_old is None:
                new_strips.append(s_new)
            else:
                blended = alpha * s_old + (1.0 - alpha) * s_new
                norm    = np.linalg.norm(blended)
                new_strips.append(blended / (norm + 1e-8))

        self._gallery[pid]["strips"]    = new_strips
        self._gallery[pid]["last_seen"] = self._frame_idx
        self._gallery[pid]["n_obs"]     = self._gallery[pid].get("n_obs", 0) + 1

    def _best_match(
        self,
        query_strips: List[Optional[np.ndarray]],
        threshold:    float,
        exclude:      Optional[set] = None,
    ) -> Tuple[Optional[int], float]:
        best_pid:  Optional[int] = None
        best_dist: float         = threshold
        for pid, entry in self._gallery.items():
            if exclude and pid in exclude:
                continue
            dist = self._weighted_distance(query_strips, entry["strips"])
            if dist is not None and dist < best_dist:
                best_dist = dist
                best_pid  = pid
        return best_pid, best_dist

    def update(
        self,
        frame_bgr:   np.ndarray,
        tracker_ids: List[int],
        boxes:       List[Tuple[float, float, float, float]],
    ) -> Dict[int, str]:
        self._frame_idx += 1
        labels:        Dict[int, str] = {}
        assigned_pids: set            = set()

        for tid, box in zip(tracker_ids, boxes):
            strips, quality = self._extract_strips(frame_bgr, box)
            pid = self._resolve(tid, strips, quality, assigned_pids)
            assigned_pids.add(pid)
            labels[tid] = "Person {}".format(pid)

        return labels

    def _resolve(
        self,
        tid:           int,
        strips:        List[Optional[np.ndarray]],
        quality:       float,
        assigned_pids: set,
    ) -> int:
        if tid in self._tracker_to_persistent:
            cached_pid = self._tracker_to_persistent[tid]

            if cached_pid in self._gallery:
                dist = self._weighted_distance(
                    strips, self._gallery[cached_pid]["strips"]
                )
                if dist is not None and dist > _REASSIGNMENT_DIST:
                    del self._tracker_to_persistent[tid]
                    return self._match_or_register(
                        tid, strips, quality, assigned_pids
                    )

            if cached_pid not in assigned_pids:
                if (quality >= _MIN_UPDATE_QUALITY
                        and cached_pid in self._gallery):
                    self._update_gallery(cached_pid, strips, quality)
                self._gallery.setdefault(cached_pid, {
                    "strips": strips, "last_seen": self._frame_idx, "n_obs": 1
                })["last_seen"] = self._frame_idx
                return cached_pid
            else:
                return self._match_or_register(
                    tid, strips, quality, assigned_pids
                )

        return self._match_or_register(tid, strips, quality, assigned_pids)

    def _match_or_register(
        self,
        tid:           int,
        strips:        List[Optional[np.ndarray]],
        quality:       float,
        assigned_pids: set,
    ) -> int:
        pid, dist = self._best_match(strips, self.threshold, exclude=assigned_pids)
        if pid is not None:
            if quality >= _MIN_UPDATE_QUALITY:
                self._update_gallery(pid, strips, quality)
            self._tracker_to_persistent[tid] = pid
            return pid

        pid, dist = self._best_match(
            strips, self.threshold_relax, exclude=assigned_pids
        )
        if pid is not None:
            if quality >= _MIN_UPDATE_QUALITY * 2:   # higher bar for relaxed match
                self._update_gallery(pid, strips, quality)
            self._tracker_to_persistent[tid] = pid
            return pid

        pid = self._register_new_person(strips)
        self._tracker_to_persistent[tid] = pid
        return pid

    def get_persistent_id(self, tracker_id: int) -> Optional[int]:
        return self._tracker_to_persistent.get(tracker_id)

    def reset(self) -> None:
        self._gallery.clear()
        self._tracker_to_persistent.clear()
        self._next_pid  = 1
        self._frame_idx = 0

print("✅  HSVPersonReIdentifier  ready")

In [ ]:
VIDEO_FOR_REID = "test_video.mp4"
_out_path = "test_video_hsv.mp4"
if os.path.exists(VIDEO_FOR_REID_EVAL):
    _demo = BaseDemo(
        video_path  = VIDEO_FOR_REID,
        output_path = _out_path,
        enable_reid = True,
        visual_threshold = 0.5,
        detect_rate = 4,
    )
    _demo.run(reid_cls=HSVPersonReIdentifier) 

In [ ]:
if os.path.exists(_out_path):
    NotebookUtils.show_video(_out_path)
    NotebookUtils.compare_videos(
        videos=[
            {"path": VIDEO_FOR_REID, "label": "Raw Input"},
            {"path": _out_path, "label": "HSV BRE-Entry Test Case"},
        ],
        n_frames=5,
        title="Method 1 — HSV Baseline Re-Entry",
           )

In [ ]:
os.chdir(str(DEMO_DIR))
VIDEO_FOR_REID_EVAL = "baber_shop.mp4"

print("Extracting tracks from:", VIDEO_FOR_REID_EVAL)
_frames_hsv, _tracks_hsv = extract_tracks_from_video(
    VIDEO_FOR_REID_EVAL, min_track_frames=25, max_frames=400
)
print(f"  → {len(_frames_hsv)} frames, {len(_tracks_hsv)} tracks")

_reid_hsv   = HSVPersonReIdentifier(use_bg_crop=True, use_strips=True,
                                     use_occlusion=True, use_quality_ema=True)
_result_hsv = evaluate_config(_frames_hsv, _tracks_hsv, _reid_hsv,
                               config_name="HSV Baseline", seed=42)

In [ ]:
_all_results = [_result_hsv]
show_reid_results(_all_results, title="Method 1 — HSV Baseline")

In [ ]:
_out_path = "output_reid_hsv.mp4"
if os.path.exists(VIDEO_FOR_REID_EVAL):
    _demo = BaseDemo(
        video_path  = VIDEO_FOR_REID_EVAL,
        output_path = _out_path,
        enable_reid = True,
        visual_threshold = 0.5,
        detect_rate = 4,
    )
    _demo.run(reid_cls=HSVPersonReIdentifier) 

In [ ]:
if os.path.exists(_out_path):
    NotebookUtils.show_video(_out_path)
    NotebookUtils.compare_videos(
        videos=[
            {"path": VIDEO_FOR_REID_EVAL, "label": "Raw Input"},
            {"path": _out_path, "label": "HSV Baseline Re-ID"},
        ],
        n_frames=5,
        title="Method 1 — HSV Baseline",
    )

# ❄️ METHOD 2 — ResNet50 Deep Features (ImageNet pre-trained)

**How it works:** Replaces the HSV histogram with a 2048-dim feature vector extracted
by a ResNet50 backbone (final FC removed, L2-normalised).  
Uses **dual-anchor gallery** (permanent anchor + EMA-updated recent) to prevent drift,
and **active tracker lock** to prevent ID theft while a person is in frame.

**Same-person cosine distance:** ~0.05–0.25  
**Different-person cosine distance:** ~0.40–0.90  
Decision boundary: strict=0.30, relaxed=0.50

**Strengths:** Much more discriminative than HSV. Robust to pose/partial occlusion.  
**Weakness:** Generic ImageNet features — not tuned for Re-ID or this specific environment.


In [ ]:
from __future__ import annotations

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from typing import Dict, List, Optional, Tuple

PERSON_COLORS_RGB = (
    ( 46, 204, 113),   # Person  1 - green
    ( 52, 152, 219),   # Person  2 - blue
    (155,  89, 182),   # Person  3 - purple
    (241, 196,  15),   # Person  4 - yellow
    ( 26, 188, 156),   # Person  5 - teal
    (230, 126,  34),   # Person  6 - orange
    (231,  76,  60),   # Person  7 - red
    ( 52,  73,  94),   # Person  8 - dark slate
    (149, 165, 166),   # Person  9 - silver
    (211,  84,   0),   # Person 10 - burnt orange
    ( 39, 174,  96),   # Person 11 - dark green
    (142,  68, 173),   # Person 12 - dark purple
)

def person_color_rgb(persistent_id):
    return PERSON_COLORS_RGB[(persistent_id - 1) % len(PERSON_COLORS_RGB)]

_REID_H          = 256      # crop height for ResNet50
_REID_W          = 128      # crop width  for ResNet50
_BG_MARGIN       = 0.10     # fraction of box width to crop each side

LOCK_RELEASE_FRAMES = 12

_MIN_UPDATE_QUAL = 0.008

THRESHOLD_STRICT = 0.30   # high-confidence match
THRESHOLD_RELAX  = 0.50   # re-entry / lighting change match

_ANCHOR_WEIGHT = 0.40
_RECENT_WEIGHT = 0.60

_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD  = [0.229, 0.224, 0.225]

_H_BINS = 32
_S_BINS = 32

class _ResNet50Extractor(nn.Module):
    def __init__(self, device):
        super(_ResNet50Extractor, self).__init__()
        resnet        = models.resnet50(pretrained=True)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.device   = device
        self.to(device)
        self.eval()
        self.preprocess = T.Compose([
            T.ToPILImage(),
            T.Resize((_REID_H, _REID_W)),
            T.ToTensor(),
            T.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
        ])

    @torch.no_grad()
    def extract_batch(self, bgr_crops):
        if not bgr_crops:
            return np.zeros((0, 2048), dtype=np.float32)
        tensors = []
        for crop in bgr_crops:
            if crop is None or crop.size == 0 \
                    or crop.shape[0] < 4 or crop.shape[1] < 4:
                tensors.append(torch.zeros(3, _REID_H, _REID_W))
            else:
                rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                tensors.append(self.preprocess(rgb))
        batch = torch.stack(tensors).to(self.device)
        feats = self.backbone(batch).flatten(1)
        feats = F.normalize(feats, p=2, dim=1)
        return feats.cpu().numpy().astype(np.float32)

def _hsv_histogram(bgr_crop):
    if bgr_crop is None or bgr_crop.size == 0:
        return np.zeros(_H_BINS * _S_BINS, dtype=np.float32)
    hsv  = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist(
        [hsv], [0, 1], None, [_H_BINS, _S_BINS], [0, 180, 0, 256]
    ).flatten().astype(np.float32)
    norm = np.linalg.norm(hist)
    return hist / (norm + 1e-8)

class PersonReIdentifier:
    def __init__(
        self,
        device                = None,
        reid_threshold        = THRESHOLD_STRICT,
        reid_threshold_relax  = THRESHOLD_RELAX,
        ema_alpha_near        = 0.85,
        ema_alpha_far         = 0.97,
        max_gallery_size      = 64,
        lock_release_frames   = LOCK_RELEASE_FRAMES,
        use_deep_features     = True,
        use_bg_crop           = True,
        use_quality_ema       = True,
        use_strips            = True,
        use_occlusion         = True,
        h_bins                = 32,
        s_bins                = 32,
    ):
        if device is None:
            device = torch.device(
                "cuda" if torch.cuda.is_available() else "cpu"
            )

        self.device               = device
        self.threshold            = reid_threshold
        self.threshold_relax      = reid_threshold_relax
        self.alpha_near           = ema_alpha_near
        self.alpha_far            = ema_alpha_far
        self.max_gallery_size     = max_gallery_size
        self.lock_release_frames  = lock_release_frames
        self.use_deep_features    = use_deep_features
        self.use_bg_crop          = use_bg_crop
        self.use_quality_ema      = use_quality_ema

        if use_deep_features:
            print("  [Re-ID] Loading ResNet50 on {} ...".format(device))
            self._extractor = _ResNet50Extractor(device)
            print("  [Re-ID] ResNet50 ready.")
        else:
            self._extractor = None

        self._gallery               = {}   # type: Dict[int, Dict]

        self._tracker_to_pid        = {}   # type: Dict[int, int]

        self._pid_to_tracker        = {}   # type: Dict[int, int]

        self._tid_last_seen         = {}   # type: Dict[int, int]

        self._next_pid  = 1
        self._frame_idx = 0

    def _get_crop(self, frame_bgr, box):
        H, W = frame_bgr.shape[:2]
        x1 = max(0, int(box[0]));  y1 = max(0, int(box[1]))
        x2 = min(W, int(box[2]));  y2 = min(H, int(box[3]))
        bw = max(1, x2 - x1);     bh = max(1, y2 - y1)
        if self.use_bg_crop:
            m = int(bw * _BG_MARGIN)
            cx1 = x1 + m;  cx2 = x2 - m
            if cx2 <= cx1 + 4:
                cx1, cx2 = x1, x2
        else:
            cx1, cx2 = x1, x2
        crop    = frame_bgr[y1:y2, cx1:cx2]
        quality = (bw * bh) / float(max(1, H * W))
        return crop, quality

    def _extract_batch(self, frame_bgr, boxes):
        crops, qualities = [], []
        for box in boxes:
            c, q = self._get_crop(frame_bgr, box)
            crops.append(c)
            qualities.append(q)
        if self.use_deep_features:
            batch_feats = self._extractor.extract_batch(crops)
            feats = [batch_feats[i] for i in range(len(crops))]
        else:
            feats = [_hsv_histogram(c) for c in crops]
        return feats, qualities

    @staticmethod
    def _cos_dist(a, b):
        return float(1.0 - np.dot(a, b))

    def _gallery_dist(self, feat, pid):
        entry = self._gallery[pid]
        d_anchor = self._cos_dist(feat, entry["anchor_feat"])
        d_recent = self._cos_dist(feat, entry["recent_feat"])
        return _ANCHOR_WEIGHT * d_anchor + _RECENT_WEIGHT * d_recent

    def _register(self, feat, tid):
        if len(self._gallery) >= self.max_gallery_size:
            candidates = {
                k: v for k, v in self._gallery.items()
                if v["active_tid"] is None
            }
            if candidates:
                oldest = min(candidates,
                             key=lambda k: self._gallery[k]["last_frame"])
                del self._gallery[oldest]

        pid = self._next_pid
        self._next_pid += 1
        self._gallery[pid] = {
            "anchor_feat": feat.copy(),  # permanent — never changed
            "recent_feat": feat.copy(),  # EMA updated
            "active_tid":  tid,
            "last_frame":  self._frame_idx,
            "n_obs":       1,
        }
        return pid

    def _ema_update(self, pid, feat, quality):
        if self.use_quality_ema:
            alpha = self.alpha_near if quality >= 0.01 else self.alpha_far
        else:
            alpha = self.alpha_near

        entry = self._gallery[pid]

        if quality < _MIN_UPDATE_QUAL:
            return

        d = self._cos_dist(feat, entry["recent_feat"])
        if d > 0.45:  # appearance changed too much in one frame
            return

        blended = alpha * entry["recent_feat"] + (1.0 - alpha) * feat
        norm    = np.linalg.norm(blended)
        entry["recent_feat"] = blended / (norm + 1e-8)
        entry["last_frame"]  = self._frame_idx
        entry["n_obs"]       = entry.get("n_obs", 0) + 1

    def _best_match(self, feat, threshold, exclude=None):
        best_pid, best_dist = None, threshold
        for pid, entry in self._gallery.items():
            if exclude and pid in exclude:
                continue
            if entry.get("active_tid") is not None:
                continue
            dist = self._gallery_dist(feat, pid)
            if dist < best_dist:
                best_dist = dist
                best_pid  = pid
        return best_pid, best_dist

    def _refresh_locks(self, current_tids):
        current_tids_set = set(current_tids)

        for pid, entry in self._gallery.items():
            tid = entry.get("active_tid")
            if tid is None:
                continue  # already released

            if tid in current_tids_set:
                continue

            last = self._tid_last_seen.get(tid, 0)
            frames_absent = self._frame_idx - last

            if frames_absent >= self.lock_release_frames:
                entry["active_tid"] = None
                if tid in self._tracker_to_pid:
                    del self._tracker_to_pid[tid]
                if self._pid_to_tracker.get(pid) == tid:
                    del self._pid_to_tracker[pid]

    def _resolve(self, tid, feat, quality, assigned_pids):
        if tid in self._tracker_to_pid:
            pid = self._tracker_to_pid[tid]

            if pid in self._gallery:
                d = self._gallery_dist(feat, pid)
                if d > 0.65:
                    self._gallery[pid]["active_tid"] = None
                    del self._tracker_to_pid[tid]
                    if self._pid_to_tracker.get(pid) == tid:
                        del self._pid_to_tracker[pid]
                    return self._match_or_register(
                        tid, feat, quality, assigned_pids
                    )

            if pid not in assigned_pids:
                self._ema_update(pid, feat, quality)
                self._gallery[pid]["active_tid"]  = tid
                self._gallery[pid]["last_frame"]  = self._frame_idx
                self._tid_last_seen[tid]          = self._frame_idx
                return pid
            else:
                return self._match_or_register(
                    tid, feat, quality, assigned_pids
                )

        return self._match_or_register(tid, feat, quality, assigned_pids)

    def _match_or_register(self, tid, feat, quality, assigned_pids):
        pid, _ = self._best_match(feat, self.threshold, exclude=assigned_pids)
        if pid is not None:
            self._ema_update(pid, feat, quality)
            self._gallery[pid]["active_tid"] = tid
            self._gallery[pid]["last_frame"] = self._frame_idx
            self._tracker_to_pid[tid]        = pid
            self._pid_to_tracker[pid]        = tid
            self._tid_last_seen[tid]         = self._frame_idx
            return pid

        pid, _ = self._best_match(
            feat, self.threshold_relax, exclude=assigned_pids
        )
        if pid is not None:
            if quality >= _MIN_UPDATE_QUAL * 2:
                self._ema_update(pid, feat, quality)
            self._gallery[pid]["active_tid"] = tid
            self._gallery[pid]["last_frame"] = self._frame_idx
            self._tracker_to_pid[tid]        = pid
            self._pid_to_tracker[pid]        = tid
            self._tid_last_seen[tid]         = self._frame_idx
            return pid

        pid = self._register(feat, tid)
        self._tracker_to_pid[tid]  = pid
        self._pid_to_tracker[pid]  = tid
        self._tid_last_seen[tid]   = self._frame_idx
        return pid

    def update(self, frame_bgr, tracker_ids, boxes):
        self._frame_idx += 1

        if not tracker_ids:
            return {}

        for tid in tracker_ids:
            self._tid_last_seen[tid] = self._frame_idx

        self._refresh_locks(tracker_ids)

        feats, qualities = self._extract_batch(frame_bgr, boxes)

        labels        = {}
        assigned_pids = set()

        for tid, feat, quality in zip(tracker_ids, feats, qualities):
            pid = self._resolve(tid, feat, quality, assigned_pids)
            assigned_pids.add(pid)
            labels[tid] = "Person {}".format(pid)

        return labels

    def get_persistent_id(self, tracker_id):
        return self._tracker_to_pid.get(tracker_id)

    def reset(self):
        self._gallery.clear()
        self._tracker_to_pid.clear()
        self._pid_to_tracker.clear()
        self._tid_last_seen.clear()
        self._next_pid  = 1
        self._frame_idx = 0

ResNet50PersonReIdentifier = PersonReIdentifier
print("✅  ResNet50PersonReIdentifier  ready (ImageNet weights)")

In [ ]:
import os, torch as _t
os.chdir(str(DEMO_DIR))

_reid_resnet = ResNet50PersonReIdentifier(
    device            = _t.device("cuda" if _t.cuda.is_available() else "cpu"),
    use_deep_features = True,
    use_bg_crop       = True,
    use_quality_ema   = True,
)

In [ ]:
_result_resnet = evaluate_config(_frames_hsv, _tracks_hsv, _reid_resnet, config_name="ResNet50 Base", seed=42)

_all_results.append(_result_resnet)
show_reid_results(_all_results, title="Methods 1–2 Comparison")

In [ ]:
_out_path = "output_reid_resnet50.mp4"
if os.path.exists(VIDEO_FOR_REID_EVAL):
    _demo = BaseDemo(
        video_path  = VIDEO_FOR_REID_EVAL,
        output_path = _out_path,
        enable_reid = True,
        visual_threshold = 0.5,
        detect_rate = 4,
    )
    _demo.run()

In [ ]:
if os.path.exists(_out_path):
    NotebookUtils.show_video(_out_path)
    NotebookUtils.compare_videos(
        videos=[
            {"path": VIDEO_FOR_REID_EVAL, "label": "Raw Input"},
            {"path": _out_path, "label": "ResNet50 Base Re-ID"},
        ],
        n_frames=5,
        title="Method 2 — ResNet50 Base",
    )

# ❄️ METHOD 3 — ResNet50 Fine-Tuned on Target Video

**How it works:** Fine-tunes the ResNet50 backbone on the target video using
**self-supervised tracklet metric learning** — no manual labels required.

The YOLO tracker provides free supervision:
- **Positive pair:** Same tracker_id within a short temporal window → same person
- **Negative pair:** Different tracker_ids visible in the same frame → different people

Training uses **batch-hard triplet loss** (Hermans et al., 2017) with online hard mining.  
Only the last 25% of the network (`layer4 + avgpool`) is fine-tuned; earlier layers are frozen.

**Why it matters:** Pushes same-person distance down to ~0.02–0.10, different-person distance
up to ~0.60–0.95. The decision margin roughly **triples** compared to vanilla ImageNet ResNet50.

**Training time:** ~5–10 minutes on an A100 GPU.


In [ ]:
from __future__ import annotations

import argparse
import random
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

class ReIDBackbone(nn.Module):
    def __init__(self, pretrained_path: Optional[str] = None):
        super().__init__()
        base = models.resnet50(pretrained=(pretrained_path is None))
        self.frozen = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3,
        )
        self.trainable = nn.Sequential(base.layer4, base.avgpool)
        self.embed_dim = 2048

        for p in self.frozen.parameters():
            p.requires_grad = False

        if pretrained_path and Path(pretrained_path).exists():
            state = torch.load(pretrained_path, map_location="cpu")
            self.load_state_dict(state, strict=False)
            print(f"  Loaded weights from: {pretrained_path}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            x = self.frozen(x)
        x = self.trainable(x)
        x = x.flatten(1)
        return F.normalize(x, p=2, dim=1)

    def trainable_parameters(self):
        return self.trainable.parameters()

class TripletLossHardMining(nn.Module):
    def __init__(self, margin: float = 0.30):
        super().__init__()
        self.margin = margin

    def forward(
        self,
        embeddings: torch.Tensor,   # (N, D)
        labels:     torch.Tensor,   # (N,) integer identity labels
    ) -> Tuple[torch.Tensor, dict]:
        N = embeddings.size(0)
        sim   = embeddings @ embeddings.T          # (N, N)
        dist  = (1.0 - sim).clamp(min=0.0)         # cosine dist

        same  = (labels.unsqueeze(0) == labels.unsqueeze(1))  # (N, N)
        diff  = ~same
        eye   = torch.eye(N, dtype=torch.bool, device=embeddings.device)
        same  = same & ~eye

        loss   = torch.tensor(0.0, device=embeddings.device)
        n_trip = 0

        for i in range(N):
            pos_mask = same[i]
            neg_mask = diff[i]
            if not pos_mask.any() or not neg_mask.any():
                continue
            hardest_pos = dist[i][pos_mask].max()
            hardest_neg = dist[i][neg_mask].min()
            trip = (hardest_pos - hardest_neg + self.margin).clamp(min=0.0)
            loss = loss + trip
            n_trip += 1

        loss = loss / max(n_trip, 1)

        pos_dists = dist[same].detach()
        neg_dists = dist[diff].detach()
        stats = {
            "loss":     loss.item(),
            "pos_mean": pos_dists.mean().item() if pos_dists.numel() > 0 else 0.0,
            "neg_mean": neg_dists.mean().item() if neg_dists.numel() > 0 else 0.0,
            "margin":   (neg_dists.mean() - pos_dists.mean()).item()
                        if (pos_dists.numel() > 0 and neg_dists.numel() > 0) else 0.0,
            "n_triplets": n_trip,
        }
        return loss, stats

class TrackletCrop:
    __slots__ = ("bgr", "label")
    def __init__(self, bgr: np.ndarray, label: int):
        self.bgr   = bgr
        self.label = label

def _iou(a, b) -> float:
    ax1,ay1,ax2,ay2 = a; bx1,by1,bx2,by2 = b
    ix1=max(ax1,bx1); iy1=max(ay1,by1)
    ix2=min(ax2,bx2); iy2=min(ay2,by2)
    inter = max(0.0,ix2-ix1)*max(0.0,iy2-iy1)
    ua    = (ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
    return inter/ua if ua>0 else 0.0

def extract_tracklet_crops(
    video_path:     str,
    max_frames:     int   = 800,
    window_frames:  int   = 20,    # reliable same-person window length
    min_crops_per_id: int = 6,     # minimum crops to keep an identity
    crop_h:         int   = 256,
    crop_w:         int   = 128,
    bg_margin:      float = 0.08,  # fraction to crop from each side
    min_area_frac:  float = 0.004, # min box area as fraction of frame
    min_height_frac:float = 0.10,  # min box height as fraction of frame H
    verbose:        bool  = True,
) -> List[TrackletCrop]:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open: {video_path}")

    W  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fgbg     = cv2.createBackgroundSubtractorMOG2(200, 50, False)
    min_area = min_area_frac * W * H
    min_h    = min_height_frac * H
    max_age  = 8

    active: Dict[int, dict]   = {}   # tid -> {box, last}
    crops:  Dict[int, List]   = {}   # tid -> [TrackletCrop, ...]
    next_tid = 1
    label_counter = 0     # sequential identity label for training

    for frame_idx in range(max_frames):
        ok, frame = cap.read()
        if not ok:
            break

        fg = fgbg.apply(frame)
        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN,
             cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)))
        fg = cv2.dilate(fg, np.ones((15,15), np.uint8), iterations=2)

        cnts, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cands = []
        for c in cnts:
            x,y,w,h = cv2.boundingRect(c)
            if w*h < min_area: continue
            if h < min_h:      continue
            if h < w:          continue
            cands.append((float(x),float(y),float(x+w),float(y+h)))

        used_c = set()
        for tid, ts in list(active.items()):
            best_iou, best_ci = 0.0, -1
            for ci, cb in enumerate(cands):
                if ci in used_c: continue
                v = _iou(ts["box"], cb)
                if v > best_iou: best_iou,best_ci = v,ci
            if best_iou > 0.15 and best_ci >= 0:
                cb = cands[best_ci]
                ts["box"] = cb; ts["last"] = frame_idx
                crop = _extract_crop(frame, cb, H, W, crop_h, crop_w, bg_margin)
                if crop is not None:
                    crops.setdefault(tid, []).append(crop)
                used_c.add(best_ci)

        for ci, cb in enumerate(cands):
            if ci not in used_c:
                tid = next_tid; next_tid += 1
                active[tid] = {"box": cb, "last": frame_idx}
                crop = _extract_crop(frame, cb, H, W, crop_h, crop_w, bg_margin)
                if crop is not None:
                    crops.setdefault(tid, []).append(crop)

        stale = [t for t,ts in active.items() if frame_idx-ts["last"]>max_age]
        for t in stale:
            del active[t]

    cap.release()

    result: List[TrackletCrop] = []
    for tid, crop_list in crops.items():
        if len(crop_list) < min_crops_per_id:
            continue
        lbl = label_counter
        label_counter += 1
        for crop in crop_list:
            result.append(TrackletCrop(bgr=crop, label=lbl))

    if verbose:
        n_ids = label_counter
        print(f"    {Path(video_path).name}: {n_ids} identities, "
              f"{len(result)} total crops ({max_frames} frames scanned)")
    return result

def _extract_crop(
    frame: np.ndarray, box: Tuple, H: int, W: int,
    crop_h: int, crop_w: int, bg_margin: float
) -> Optional[np.ndarray]:
    x1,y1,x2,y2 = box
    bw = max(1, x2 - x1)
    m  = int(bw * bg_margin)
    cx1 = max(0, int(x1) + m)
    cx2 = min(W, int(x2) - m)
    cy1 = max(0, int(y1))
    cy2 = min(H, int(y2))
    if cx2 - cx1 < 8 or cy2 - cy1 < 16:
        return None
    crop = frame[cy1:cy2, cx1:cx2]
    if crop.size == 0:
        return None
    return cv2.resize(crop, (crop_w, crop_h))

class PKBatchSampler:
    def __init__(self, labels: List[int], P: int = 8, K: int = 8,
                 n_batches: int = 200):
        self.P = P; self.K = K; self.n_batches = n_batches
        self.id_to_idx: Dict[int, List[int]] = {}
        for i, lbl in enumerate(labels):
            self.id_to_idx.setdefault(lbl, []).append(i)
        self.valid_ids = [l for l,idxs in self.id_to_idx.items() if len(idxs) >= 2]
        if len(self.valid_ids) < P:
            self.P = len(self.valid_ids)

    def __iter__(self):
        for _ in range(self.n_batches):
            chosen_ids = random.sample(self.valid_ids, min(self.P, len(self.valid_ids)))
            batch_idx  = []
            for lbl in chosen_ids:
                pool = self.id_to_idx[lbl]
                picks = random.choices(pool, k=self.K)  # with replacement if needed
                batch_idx.extend(picks)
            yield batch_idx

    def __len__(self):
        return self.n_batches

_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_transform() -> T.Compose:
    return T.Compose([
        T.ToPILImage(),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        T.RandomGrayscale(p=0.1),          # simulate low-colour thermal
        T.RandomAffine(degrees=8, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        T.Resize((256, 128)),
        T.RandomCrop((256, 128), padding=8),
        T.ToTensor(),
        T.Normalize(_IMAGENET_MEAN, _IMAGENET_STD),
        T.RandomErasing(p=0.3, scale=(0.02, 0.15)),  # simulate occlusion
    ])

def get_eval_transform() -> T.Compose:
    return T.Compose([
        T.ToPILImage(),
        T.Resize((256, 128)),
        T.ToTensor(),
        T.Normalize(_IMAGENET_MEAN, _IMAGENET_STD),
    ])

def evaluate_distances(
    model:   ReIDBackbone,
    crops:   List[TrackletCrop],
    device:  torch.device,
    n_eval:  int = 200,
) -> Tuple[float, float, float]:
    tf     = get_eval_transform()
    model.eval()
    rng    = random.Random(0)

    id_to_crops: Dict[int, List[TrackletCrop]] = {}
    for c in crops:
        id_to_crops.setdefault(c.label, []).append(c)
    ids = [l for l,cs in id_to_crops.items() if len(cs) >= 2]
    if not ids:
        return 0.0, 0.0, 0.0

    pos_dists, neg_dists = [], []
    with torch.no_grad():
        for _ in range(n_eval):
            lbl  = rng.choice(ids)
            a, b = rng.sample(id_to_crops[lbl], 2)
            ta   = tf(cv2.cvtColor(a.bgr, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(device)
            tb   = tf(cv2.cvtColor(b.bgr, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(device)
            ea   = model(ta); eb = model(tb)
            pos_dists.append(float(1.0 - (ea * eb).sum()))

            other = rng.choice([l for l in ids if l != lbl])
            c     = rng.choice(id_to_crops[other])
            tc    = tf(cv2.cvtColor(c.bgr, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(device)
            ec    = model(tc)
            neg_dists.append(float(1.0 - (ea * ec).sum()))

    mu_pos = float(np.mean(pos_dists))
    mu_neg = float(np.mean(neg_dists))
    return mu_pos, mu_neg, mu_neg - mu_pos

def train_reid(
    crops:       List[TrackletCrop],
    out_path:    str,
    epochs:      int   = 15,
    lr:          float = 1e-4,
    margin:      float = 0.30,
    P:           int   = 8,
    K:           int   = 8,
    n_batches:   int   = 200,
    device:      torch.device = None,
    pretrained_path: Optional[str] = None,
) -> None:
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"\n  Training on {device}")
    print(f"  Crops: {len(crops)}")

    labels = [c.label for c in crops]
    n_ids  = len(set(labels))
    print(f"  Identities: {n_ids}")
    print(f"  Batch: P={P} identities × K={K} crops = {P*K} samples")
    print(f"  Epochs: {epochs}  ×  {n_batches} batches/epoch")
    print(f"  Triplet margin: {margin}")

    if n_ids < 4:
        print("\n  ERROR: Need at least 4 identities to train. "
              "Use longer videos or lower --min-crops.")
        return

    model     = ReIDBackbone(pretrained_path=pretrained_path).to(device)
    criterion = TripletLossHardMining(margin=margin)
    optimiser = Adam(model.trainable_parameters(), lr=lr, weight_decay=5e-4)
    scheduler = CosineAnnealingLR(optimiser, T_max=epochs, eta_min=lr * 0.01)

    tf      = get_train_transform()
    sampler = PKBatchSampler(labels, P=P, K=K, n_batches=n_batches)

    print("\n  Baseline (before fine-tuning):")
    mu_pos0, mu_neg0, margin0 = evaluate_distances(model, crops, device)
    print(f"    same-person dist: {mu_pos0:.4f}  "
          f"diff-person dist: {mu_neg0:.4f}  "
          f"margin: {margin0:.4f}")

    best_margin = margin0
    best_state  = None
    t0          = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss   = 0.0
        epoch_margin = 0.0
        n_batches_run = 0

        for batch_indices in sampler:
            imgs_list  = []
            lbls_list  = []
            for idx in batch_indices:
                crop = crops[idx]
                rgb  = cv2.cvtColor(crop.bgr, cv2.COLOR_BGR2RGB)
                imgs_list.append(tf(rgb))
                lbls_list.append(crop.label)
            imgs = torch.stack(imgs_list).to(device)
            lbls = torch.tensor(lbls_list, device=device)

            embeddings = model(imgs)
            loss, stats = criterion(embeddings, lbls)

            optimiser.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.trainable_parameters(), 1.0)
            optimiser.step()

            epoch_loss   += stats["loss"]
            epoch_margin += stats["margin"]
            n_batches_run += 1

        scheduler.step()

        avg_loss   = epoch_loss   / max(n_batches_run, 1)
        avg_margin = epoch_margin / max(n_batches_run, 1)
        elapsed    = time.time() - t0

        if epoch % 3 == 0 or epoch == epochs:
            mu_pos, mu_neg, cur_margin = evaluate_distances(model, crops, device)
            marker = " ← best" if cur_margin > best_margin else ""
            print(f"  Epoch {epoch:>2}/{epochs}  "
                  f"loss={avg_loss:.4f}  "
                  f"pos={mu_pos:.4f}  neg={mu_neg:.4f}  "
                  f"margin={cur_margin:.4f}{marker}  "
                  f"({elapsed:.0f}s)")
            if cur_margin > best_margin:
                best_margin = cur_margin
                best_state  = {k: v.cpu().clone()
                                for k, v in model.state_dict().items()}
        else:
            print(f"  Epoch {epoch:>2}/{epochs}  "
                  f"loss={avg_loss:.4f}  "
                  f"margin≈{avg_margin:.4f}  "
                  f"({elapsed:.0f}s)")

    if best_state is None:
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(best_state, out_path)

    print(f"\n  ✓  Fine-tuned weights saved → {out_path}")
    print(f"  Margin improvement: {margin0:.4f} → {best_margin:.4f} "
          f"(+{best_margin-margin0:.4f}, "
          f"{(best_margin-margin0)/max(margin0,1e-6)*100:.0f}% better)")
    print(f"\n  To use in demo.py, set in PersonReIdentifier init:")
    print(f"      weight_path = '{out_path}'")

print("✅  TrackletFineTuner  ready")

In [ ]:
import torch

FinetunedPersonReIdentifier = ResNet50PersonReIdentifier

print("✅  FinetunedPersonReIdentifier  ready")
print("   Uses ResNet50PersonReIdentifier with optional fine-tuned weight_path")

In [ ]:
import torch as _t
os.chdir(str(DEMO_DIR))

FINETUNE_OUT = "reid_finetuned.pth"

if not os.path.exists(FINETUNE_OUT):
    print("Fine-tuning ResNet50 on test_video.mp4 ...")
    trainer = TrackletFineTuner(
        video_paths = ["test_video.mp4"],
        device      = _t.device("cuda" if _t.cuda.is_available() else "cpu"),
    )
    trainer.run(epochs=10, out_path=FINETUNE_OUT)
else:
    print(f"Fine-tuned weights already exist: {FINETUNE_OUT}")


In [ ]:
_reid_ft = FinetunedPersonReIdentifier(
    device            = _t.device("cuda" if _t.cuda.is_available() else "cpu"),
    weight_path       = FINETUNE_OUT if os.path.exists(FINETUNE_OUT) else None,
    use_deep_features = True,
    use_bg_crop       = True,
    use_quality_ema   = True,
)

In [ ]:
_result_ft = evaluate_config(_frames_hsv, _tracks_hsv, _reid_ft,
                               config_name="ResNet50 Finetune", seed=42)
_all_results.append(_result_ft)
show_reid_results(_all_results, title="Methods 1–3 Comparison")

In [ ]:
_out_path = "output_reid_finetuned.mp4"
if os.path.exists(VIDEO_FOR_REID):
    _demo = BaseDemo(
        video_path  = VIDEO_FOR_REID,
        output_path = _out_path,
        enable_reid = True,
        visual_threshold = 0.5,
        detect_rate = 4,
    )
    _demo.run()

In [ ]:
if os.path.exists(_out_path):
    NotebookUtils.show_video(_out_path)
    NotebookUtils.compare_videos(
        videos=[
            {"path": VIDEO_FOR_REID, "label": "Raw Input"},
            {"path": _out_path, "label": "ResNet50 Fine-Tuned Re-ID"},
        ],
        n_frames=5,
        title="Method 3 — ResNet50 Fine-Tuned",
    )

# ❄️ METHOD 4 - InsightFace ArcFace Re-ID (PRIMARY METHOD)

**How it works:** Uses InsightFace's ArcFace face recognition as the **primary identity signal**,
with ResNet50 appearance as an **automatic fallback** when:
- No face is detected in the crop
- The face quality score is too low (non-frontal, occluded, too small)
- The face is at an angle > 30° from frontal

**Architecture:**
Detection → Face quality gate - HIGH quality - ArcFace (512-dim) - Gallery match
LOW quality - ResNet50 (2048-dim) - Gallery match

**Key advantages over Methods 1–3:**
- ArcFace was explicitly trained for face verification — much tighter same-person clusters
- Face embeddings are identity-discriminative by design, not just appearance-similar
- Falls back gracefully to appearance when face is hidden (occlusion, back to camera)

**ArcFace properties (on LFW benchmark):**  
Same-person cosine distance: ~0.00–0.15 | Different-person: ~0.50–1.00

In [ ]:
from __future__ import annotations

import cv2 as _cv2
import numpy as _np
import torch as _torch
import torch.nn as _nn
import torch.nn.functional as _F
import torchvision.models as _tvm
import torchvision.transforms as _tvt
from typing import Dict, List, Optional, Tuple

_INSIGHTFACE_APP = None

def _get_insightface(det_size=(320, 320)):
    global _INSIGHTFACE_APP
    if _INSIGHTFACE_APP is None:
        from insightface.app import FaceAnalysis
        _INSIGHTFACE_APP = FaceAnalysis(
            name="buffalo_l",
            providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
        )
        _INSIGHTFACE_APP.prepare(ctx_id=0, det_size=det_size)
        print("  [InsightFace] ArcFace + RetinaFace loaded.")
    return _INSIGHTFACE_APP

_REID_H   = 256
_REID_W   = 128
_BG_MARG  = 0.10
_LOCK_REL = 12           # frames before tracker lock is released
_MIN_QUAL = 0.008        # min box area fraction for gallery update

_ARCFACE_STRICT  = 0.35
_ARCFACE_RELAX   = 0.55

_APPEAR_STRICT   = 0.30
_APPEAR_RELAX    = 0.50

_MIN_FACE_AREA   = 32 * 32      # minimum face box area in pixels
_MAX_POSE_DEG    = 30.0         # maximum yaw angle to trust face embedding
_MIN_DET_SCORE   = 0.65         # minimum RetinaFace detection confidence

_ANCHOR_W = 0.40
_RECENT_W = 0.60
_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD  = [0.229, 0.224, 0.225]

class _AppearanceExtractor(_nn.Module):
    def __init__(self, device, weight_path=None):
        super().__init__()
        base          = _tvm.resnet50(pretrained=(weight_path is None))
        self.backbone = _nn.Sequential(*list(base.children())[:-1])
        self.device   = device
        self.to(device)
        self.eval()
        self.preprocess = _tvt.Compose([
            _tvt.ToPILImage(),
            _tvt.Resize((_REID_H, _REID_W)),
            _tvt.ToTensor(),
            _tvt.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
        ])
        if weight_path:
            from pathlib import Path as _P
            if _P(weight_path).exists():
                sd = _torch.load(weight_path, map_location="cpu")
                self.load_state_dict(sd, strict=False)
                print(f"  [Appearance] Loaded fine-tuned weights: {weight_path}")

    @_torch.no_grad()
    def extract_batch(self, bgr_crops):
        if not bgr_crops:
            return _np.zeros((0, 2048), dtype=_np.float32)
        tensors = []
        for c in bgr_crops:
            if c is None or c.size == 0 or c.shape[0] < 4 or c.shape[1] < 4:
                tensors.append(_torch.zeros(3, _REID_H, _REID_W))
            else:
                rgb = _cv2.cvtColor(c, _cv2.COLOR_BGR2RGB)
                tensors.append(self.preprocess(rgb))
        batch = _torch.stack(tensors).to(self.device)
        feats = self.backbone(batch).flatten(1)
        return _F.normalize(feats, p=2, dim=1).cpu().numpy().astype(_np.float32)

class InsightFaceReIdentifier:
    def __init__(
        self,
        device              = None,
        weight_path         = None,     # optional fine-tuned ResNet50 weights
        face_strict         = _ARCFACE_STRICT,
        face_relax          = _ARCFACE_RELAX,
        appear_strict       = _APPEAR_STRICT,
        appear_relax        = _APPEAR_RELAX,
        ema_alpha_near      = 0.85,
        ema_alpha_far       = 0.97,
        max_gallery_size    = 64,
        lock_release_frames = _LOCK_REL,
        det_size            = (320, 320),
        use_face_primary    = True,     # False = appearance-only (ablation)
    ):
        if device is None:
            device = _torch.device("cuda" if _torch.cuda.is_available() else "cpu")
        self.device              = device
        self.face_strict         = face_strict
        self.face_relax          = face_relax
        self.appear_strict       = appear_strict
        self.appear_relax        = appear_relax
        self.alpha_near          = ema_alpha_near
        self.alpha_far           = ema_alpha_far
        self.max_gallery_size    = max_gallery_size
        self.lock_release_frames = lock_release_frames
        self.use_face_primary    = use_face_primary

        if use_face_primary:
            self._face_app = _get_insightface(det_size)
        else:
            self._face_app = None

        print(f"  [InsightFaceReID] Loading ResNet50 appearance extractor on {device}...")
        self._appear = _AppearanceExtractor(device, weight_path)
        print("  [InsightFaceReID] Ready.")

        self._gallery         : Dict[int, Dict] = {}
        self._tracker_to_pid  : Dict[int, int]  = {}
        self._pid_to_tracker  : Dict[int, int]  = {}
        self._tid_last_seen   : Dict[int, int]  = {}
        self._next_pid  = 1
        self._frame_idx = 0

        self.stats = {"face_matches": 0, "appear_matches": 0,
                      "new_persons": 0, "face_detections": 0}

    def _detect_face_in_crop(self, bgr_crop, box_in_frame, frame_h, frame_w):
        if self._face_app is None or bgr_crop is None or bgr_crop.size == 0:
            return None

        rgb_crop = _cv2.cvtColor(bgr_crop, _cv2.COLOR_BGR2RGB)
        try:
            faces = self._face_app.get(rgb_crop)
        except Exception:
            return None

        if not faces:
            return None

        best_emb  = None
        best_score = -1.0

        for face in faces:
            det_score = float(getattr(face, "det_score", 1.0))
            if det_score < _MIN_DET_SCORE:
                continue

            bbox = face.bbox.astype(int)
            fw = max(1, bbox[2] - bbox[0])
            fh = max(1, bbox[3] - bbox[1])
            if fw * fh < _MIN_FACE_AREA:
                continue

            pose = getattr(face, "pose", None)
            if pose is not None:
                yaw = float(pose[1]) if len(pose) > 1 else float(pose[0])
                if abs(yaw) > _MAX_POSE_DEG:
                    continue

            emb = getattr(face, "embedding", None)
            if emb is None:
                continue

            emb = emb.astype(_np.float32)
            norm = _np.linalg.norm(emb)
            emb  = emb / (norm + 1e-8)

            score = det_score * fw * fh
            if score > best_score:
                best_score = score
                best_emb   = emb

        if best_emb is not None:
            self.stats["face_detections"] += 1
        return best_emb

    def _get_crop(self, frame_bgr, box):
        H, W = frame_bgr.shape[:2]
        x1   = max(0, int(box[0]));  y1 = max(0, int(box[1]))
        x2   = min(W, int(box[2]));  y2 = min(H, int(box[3]))
        bw   = max(1, x2 - x1);     bh = max(1, y2 - y1)
        m    = int(bw * _BG_MARG)
        cx1  = x1 + m;              cx2 = x2 - m
        if cx2 <= cx1 + 4:
            cx1, cx2 = x1, x2
        crop    = frame_bgr[y1:y2, cx1:cx2]
        quality = (bw * bh) / float(max(1, H * W))
        return crop, quality, H, W

    @staticmethod
    def _cos(a, b):
        return float(1.0 - _np.dot(a, b))

    def _gallery_dist(self, face_emb, appear_emb, pid):
        entry = self._gallery[pid]
        has_face_q  = face_emb is not None
        has_face_g  = entry.get("has_face", False)

        if has_face_q and has_face_g:
            d_anch = self._cos(face_emb, entry["face_anchor"])
            d_rec  = self._cos(face_emb, entry["face_recent"])
            return _ANCHOR_W * d_anch + _RECENT_W * d_rec, "face"

        if appear_emb is not None and "appear_anchor" in entry:
            d_anch = self._cos(appear_emb, entry["appear_anchor"])
            d_rec  = self._cos(appear_emb, entry["appear_recent"])
            return _ANCHOR_W * d_anch + _RECENT_W * d_rec, "appear"

        return 1.0, "none"

    def _register(self, face_emb, appear_emb, tid):
        if len(self._gallery) >= self.max_gallery_size:
            candidates = {k: v for k, v in self._gallery.items()
                          if v.get("active_tid") is None}
            if candidates:
                oldest = min(candidates, key=lambda k: self._gallery[k]["last_frame"])
                del self._gallery[oldest]

        pid = self._next_pid
        self._next_pid += 1
        entry = {
            "has_face":      face_emb is not None,
            "active_tid":    tid,
            "last_frame":    self._frame_idx,
            "n_obs":         1,
        }
        if face_emb is not None:
            entry["face_anchor"] = face_emb.copy()
            entry["face_recent"] = face_emb.copy()
        if appear_emb is not None:
            entry["appear_anchor"] = appear_emb.copy()
            entry["appear_recent"] = appear_emb.copy()
        self._gallery[pid] = entry
        self.stats["new_persons"] += 1
        return pid

    def _ema_update(self, pid, face_emb, appear_emb, quality):
        entry = self._gallery[pid]
        if quality < _MIN_QUAL:
            return

        alpha = self.alpha_near if quality >= 0.01 else self.alpha_far

        if face_emb is not None:
            entry["has_face"] = True
            if "face_anchor" not in entry:
                entry["face_anchor"] = face_emb.copy()
                entry["face_recent"] = face_emb.copy()
            else:
                if self._cos(face_emb, entry["face_recent"]) < 0.40:
                    blended = alpha * entry["face_recent"] + (1 - alpha) * face_emb
                    n = _np.linalg.norm(blended)
                    entry["face_recent"] = blended / (n + 1e-8)

        if appear_emb is not None:
            if "appear_anchor" not in entry:
                entry["appear_anchor"] = appear_emb.copy()
                entry["appear_recent"] = appear_emb.copy()
            else:
                if self._cos(appear_emb, entry["appear_recent"]) < 0.45:
                    blended = alpha * entry["appear_recent"] + (1 - alpha) * appear_emb
                    n = _np.linalg.norm(blended)
                    entry["appear_recent"] = blended / (n + 1e-8)

        entry["last_frame"] = self._frame_idx
        entry["n_obs"]      = entry.get("n_obs", 0) + 1

    def _refresh_locks(self, current_tids):
        current_set = set(current_tids)
        for pid, entry in self._gallery.items():
            tid = entry.get("active_tid")
            if tid is None or tid in current_set:
                continue
            last   = self._tid_last_seen.get(tid, 0)
            absent = self._frame_idx - last
            if absent >= self.lock_release_frames:
                entry["active_tid"] = None
                self._tracker_to_pid.pop(tid, None)
                if self._pid_to_tracker.get(pid) == tid:
                    del self._pid_to_tracker[pid]

    def _best_match(self, face_emb, appear_emb, threshold, exclude=None):
        best_pid, best_dist = None, threshold
        for pid, entry in self._gallery.items():
            if exclude and pid in exclude:
                continue
            if entry.get("active_tid") is not None:
                continue
            dist, branch = self._gallery_dist(face_emb, appear_emb, pid)
            if dist < best_dist:
                best_dist = dist
                best_pid  = pid
        return best_pid, best_dist

    def _resolve(self, tid, face_emb, appear_emb, quality, assigned_pids):
        has_face = face_emb is not None
        strict   = self.face_strict  if has_face else self.appear_strict
        relax    = self.face_relax   if has_face else self.appear_relax

        if tid in self._tracker_to_pid:
            pid = self._tracker_to_pid[tid]
            if pid in self._gallery:
                dist, _ = self._gallery_dist(face_emb, appear_emb, pid)
                if dist > 0.70:
                    self._gallery[pid]["active_tid"] = None
                    self._tracker_to_pid.pop(tid, None)
                    self._pid_to_tracker.pop(pid, None)
                    return self._match_or_register(tid, face_emb, appear_emb,
                                                   quality, assigned_pids)
            if pid not in assigned_pids:
                self._ema_update(pid, face_emb, appear_emb, quality)
                self._gallery[pid]["active_tid"] = tid
                self._gallery[pid]["last_frame"] = self._frame_idx
                self._tid_last_seen[tid] = self._frame_idx
                if has_face:
                    self.stats["face_matches"] += 1
                else:
                    self.stats["appear_matches"] += 1
                return pid
            else:
                return self._match_or_register(tid, face_emb, appear_emb,
                                               quality, assigned_pids)

        return self._match_or_register(tid, face_emb, appear_emb,
                                       quality, assigned_pids)

    def _match_or_register(self, tid, face_emb, appear_emb, quality, assigned_pids):
        has_face = face_emb is not None
        strict   = self.face_strict  if has_face else self.appear_strict
        relax    = self.face_relax   if has_face else self.appear_relax

        pid, _ = self._best_match(face_emb, appear_emb, strict, exclude=assigned_pids)
        if pid is not None:
            self._ema_update(pid, face_emb, appear_emb, quality)
            self._gallery[pid]["active_tid"] = tid
            self._gallery[pid]["last_frame"] = self._frame_idx
            self._tracker_to_pid[tid] = pid
            self._pid_to_tracker[pid] = tid
            self._tid_last_seen[tid]  = self._frame_idx
            if has_face: self.stats["face_matches"] += 1
            else:        self.stats["appear_matches"] += 1
            return pid

        pid, _ = self._best_match(face_emb, appear_emb, relax, exclude=assigned_pids)
        if pid is not None:
            if quality >= _MIN_QUAL * 2:
                self._ema_update(pid, face_emb, appear_emb, quality)
            self._gallery[pid]["active_tid"] = tid
            self._gallery[pid]["last_frame"] = self._frame_idx
            self._tracker_to_pid[tid] = pid
            self._pid_to_tracker[pid] = tid
            self._tid_last_seen[tid]  = self._frame_idx
            if has_face: self.stats["face_matches"] += 1
            else:        self.stats["appear_matches"] += 1
            return pid

        pid = self._register(face_emb, appear_emb, tid)
        self._tracker_to_pid[tid] = pid
        self._pid_to_tracker[pid] = tid
        self._tid_last_seen[tid]  = self._frame_idx
        return pid

    def update(self, frame_bgr, tracker_ids, boxes):
        self._frame_idx += 1
        if not tracker_ids:
            return {}

        for tid in tracker_ids:
            self._tid_last_seen[tid] = self._frame_idx

        self._refresh_locks(tracker_ids)

        crops     = []
        qualities = []
        H_list, W_list = [], []
        for box in boxes:
            c, q, H, W = self._get_crop(frame_bgr, box)
            crops.append(c); qualities.append(q)
            H_list.append(H); W_list.append(W)

        appear_feats_batch = self._appear.extract_batch(crops)
        appear_feats = [appear_feats_batch[i] for i in range(len(crops))]

        face_feats = []
        for i, (box, crop) in enumerate(zip(boxes, crops)):
            fe = self._detect_face_in_crop(crop, box, H_list[i], W_list[i])
            face_feats.append(fe)

        labels        = {}
        assigned_pids = set()

        for tid, face_emb, appear_emb, quality in zip(
            tracker_ids, face_feats, appear_feats, qualities
        ):
            pid = self._resolve(tid, face_emb, appear_emb, quality, assigned_pids)
            assigned_pids.add(pid)
            labels[tid] = "Person {}".format(pid)

        return labels

    def get_persistent_id(self, tracker_id):
        return self._tracker_to_pid.get(tracker_id)

    def reset(self):
        self._gallery.clear()
        self._tracker_to_pid.clear()
        self._pid_to_tracker.clear()
        self._tid_last_seen.clear()
        self._next_pid  = 1
        self._frame_idx = 0
        self.stats = {"face_matches":0,"appear_matches":0,"new_persons":0,"face_detections":0}

    def print_stats(self):
        total = self.stats["face_matches"] + self.stats["appear_matches"]
        pct   = self.stats["face_matches"] / max(1, total) * 100
        print(f"  Face matches     : {self.stats['face_matches']:>6}  ({pct:.1f}% of all matches)")
        print(f"  Appear matches   : {self.stats['appear_matches']:>6}  ({100-pct:.1f}% of all matches)")
        print(f"  New persons      : {self.stats['new_persons']:>6}")
        print(f"  Face detections  : {self.stats['face_detections']:>6}")

    def person_color_rgb(self, pid):
        colors = (
            ( 46,204,113),( 52,152,219),(155, 89,182),(241,196, 15),
            ( 26,188,156),(230,126, 34),(231, 76, 60),( 52, 73, 94),
            (149,165,166),(211, 84,  0),( 39,174, 96),(142, 68,173),
        )
        return colors[(pid - 1) % len(colors)]

print("✅  InsightFaceReIdentifier  ready  (PRIMARY METHOD)")
print("   ArcFace primary + ResNet50 fallback")
print("   Face gates: det≥0.65, area≥32×32, yaw≤±30°")

In [ ]:
import os, torch as _t
os.chdir(str(DEMO_DIR))

_reid_face = InsightFaceReIdentifier(
    device       = _t.device("cuda" if _t.cuda.is_available() else "cpu"),
    weight_path  = FINETUNE_OUT if os.path.exists(FINETUNE_OUT) else None,
    use_face_primary = True,
)

_result_face = evaluate_config(_frames_hsv, _tracks_hsv, _reid_face,
                                config_name="InsightFace", seed=42)

_reid_face.print_stats()
print()
print(_result_face)

_all_results.append(_result_face)
show_reid_results(_all_results, title="All 4 Methods — Full Comparison")


In [ ]:
_out_path = "output_reid_insightface.mp4"
if os.path.exists(VIDEO_FOR_REID_EVAL):
    _demo = BaseDemo(
        video_path  = VIDEO_FOR_REID_EVAL,
        output_path = _out_path,
        enable_reid = True,
        visual_threshold = 0.5,
        detect_rate = 4,
    )
    _demo.run()

In [ ]:
if os.path.exists(_out_path):
    NotebookUtils.show_video(_out_path)
    NotebookUtils.compare_videos(
        videos=[
            {"path": VIDEO_FOR_REID_EVAL, "label": "Raw Input"},
            {"path": _out_path, "label": "InsightFace ArcFace Re-ID"},
        ],
        n_frames=5,
        title="Method 4 — InsightFace",
    )

## ▶ Run Full Pipeline with InsightFace Re-ID

Now we run `BaseDemo` with `enable_reid=True` — this plugs `InsightFaceReIdentifier`
into `AVAVisualizer` so the output video shows coloured bounding boxes with
stable **"Person N"** badges that persist across re-entries.


In [ ]:
InsightFaceDemo is no longer needed — pass reid_cls directly to run():

  demo.run(reid_cls=InsightFaceReIdentifier,
           device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
           weight_path=FINETUNE_OUT if os.path.exists(FINETUNE_OUT) else None)

Other examples:
  demo.run()                                    # no ReID, tracking only
  demo.run(reid_cls=HSVPersonReIdentifier)      # HSV baseline
  demo.run(reid_cls=ResNet50PersonReIdentifier) # ResNet50
  demo.run(reid_cls=FinetunedPersonReIdentifier, weight_path=FINETUNE_OUT)

print("✅  reid_cls injection ready — pass any ReIdentifier class to demo.run()")

In [ ]:
os.chdir(str(DEMO_DIR))

demo_face = InsightFaceDemo(
    video_path  = "test_video.mp4",
    output_path = "output_insightface_reid.mp4",
)
demo_face.run()

# 👨‍💻 Robustness Evaluation - 15 Videos × 5 Conditions

The evaluation runs across **5 conditions × 3 videos each = 15 videos** to assess how scene complexity,  
occlusion, motion blur, lighting, and crowd density affect detection and action recognition.

**Conditions**: `clean`,`crowd` ,`fast_motion`,`occlusion` & `low_light`

**Metrics collected per video**: detection rate, avg persons/frame, max persons/frame,  
total action predictions, unique actions, avg/max/min confidence, processing FPS.

In [ ]:
import time
import traceback
from dataclasses import dataclass, asdict, field
from datetime import datetime
from pathlib import Path
from time import sleep as _sleep
from itertools import count as _count
from typing import Dict, List, Optional, Tuple

import torch as _torch
from tqdm import tqdm as _tqdm
import pandas as _pd
from IPython.display import display as _display, HTML as _HTML

@dataclass
class VideoMetrics:
    condition:                str   = ""
    video_name:               str   = ""
    video_path:               str   = ""
    original_width:           int   = 0
    original_height:          int   = 0
    original_fps:             float = 0.0
    original_duration_s:      float = 0.0
    original_frames:          int   = 0
    total_frames_processed:   int   = 0
    frames_with_detections:   int   = 0
    detection_rate:           float = 0.0
    total_person_detections:  int   = 0
    avg_persons_per_frame:    float = 0.0
    max_persons_in_frame:     int   = 0
    total_action_predictions: int   = 0
    unique_actions_detected:  int   = 0
    avg_confidence:           float = 0.0
    max_confidence:           float = 0.0
    min_confidence:           float = 1.0
    actions_detected:         str   = ""
    processing_time_s:        float = 0.0
    processing_fps:           float = 0.0
    success:                  bool  = True
    error_msg:                str   = ""

@dataclass
class PredictionEvent:
    condition:    str
    video_name:   str
    timestamp_ms: float
    person_id:    int
    action:       str
    confidence:   float

class MetricsCapture:
    def __init__(self, visualizer, condition: str, video_name: str):
        self.condition  = condition
        self.video_name = video_name
        self.frame_person_counts:   List[int]             = []
        self.prediction_events:     List[PredictionEvent] = []
        self.action_confidence_all: List[float]           = []

        orig_update = visualizer._update_action_dict
        def patched_update(scores, ids):
            if scores is not None and ids is not None:
                for score, pid in zip(scores, ids):
                    above = _torch.nonzero(
                        score >= visualizer.thresh, as_tuple=False
                    ).squeeze(1).tolist()
                    for cid in above:
                        if cid in visualizer.excl_ids: continue
                        conf  = float(score[cid])
                        label = visualizer.cate_list[cid]
                        self.action_confidence_all.append(conf)
                        self.prediction_events.append(PredictionEvent(
                            condition=self.condition, video_name=self.video_name,
                            timestamp_ms=0.0, person_id=int(pid),
                            action=label, confidence=conf,
                        ))
            orig_update(scores, ids)
        visualizer._update_action_dict = patched_update

        orig_send = visualizer.send_track
        def patched_send(result):
            if isinstance(result, tuple):
                boxes, ids = result
                self.frame_person_counts.append(len(boxes) if boxes is not None else 0)
            orig_send(result)
        visualizer.send_track = patched_send

    def fill(self, vm: VideoMetrics) -> VideoMetrics:
        counts = self.frame_person_counts
        vm.total_frames_processed  = len(counts)
        vm.frames_with_detections  = sum(1 for c in counts if c > 0)
        vm.detection_rate = (vm.frames_with_detections / vm.total_frames_processed
                             if vm.total_frames_processed > 0 else 0.0)
        vm.total_person_detections = sum(counts)
        vm.avg_persons_per_frame   = (vm.total_person_detections / vm.total_frames_processed
                                      if vm.total_frames_processed > 0 else 0.0)
        vm.max_persons_in_frame    = max(counts) if counts else 0
        events = self.prediction_events
        vm.total_action_predictions = len(events)
        vm.unique_actions_detected  = len({e.action for e in events})
        vm.actions_detected         = ", ".join(sorted({e.action for e in events}))
        confs = self.action_confidence_all
        if confs:
            vm.avg_confidence = sum(confs) / len(confs)
            vm.max_confidence = max(confs)
            vm.min_confidence = min(confs)
        return vm

CONDITIONS = ["clean", "crowd", "fast_motion", "occlusion", "low_light"]
_COLORS    = ["#4C78A8", "#F58518", "#E45756", "#72B7B2", "#54A24B"]

class RobustnessEvaluator:
    def __init__(
        self,
        video_dir:   str  = "test_videos",
        output_dir:  str  = "test_results",
        enable_reid: bool = False,
        n_frames:    int  = 5,
    ):
        self.video_dir   = Path(video_dir)
        self.output_dir  = Path(output_dir)
        self.enable_reid = enable_reid
        self.n_frames    = n_frames
        self.all_metrics: List[VideoMetrics]    = []
        self.all_events:  List[PredictionEvent] = []

    def _discover(self) -> List[Tuple[str, Path]]:
        entries = []
        for cond in CONDITIONS:
            cond_dir = self.video_dir / cond
            if not cond_dir.exists():
                print(f"  [skip] missing: {cond_dir}"); continue
            videos = sorted(p for p in cond_dir.iterdir()
                            if p.suffix.lower() in {".mp4",".avi",".mov",".mkv"})
            if not videos:
                print(f"  [skip] empty: {cond_dir}"); continue
            for v in videos:
                entries.append((cond, v))
        return entries

    def _run_one(self, condition: str, video_path: Path) -> Tuple[VideoMetrics, List[PredictionEvent]]:
        import cv2 as _cv2

        video_name  = video_path.stem
        out_dir     = self.output_dir / condition
        out_dir.mkdir(parents=True, exist_ok=True)
        output_path = str(out_dir / f"{video_name}_output.mp4")

        vm = VideoMetrics(condition=condition, video_name=video_name,
                          video_path=str(video_path))

        cap = _cv2.VideoCapture(str(video_path))
        if cap.isOpened():
            fps = cap.get(_cv2.CAP_PROP_FPS)
            n   = int(cap.get(_cv2.CAP_PROP_FRAME_COUNT))
            vm.original_width      = int(cap.get(_cv2.CAP_PROP_FRAME_WIDTH))
            vm.original_height     = int(cap.get(_cv2.CAP_PROP_FRAME_HEIGHT))
            vm.original_fps        = fps
            vm.original_frames     = n
            vm.original_duration_s = n / fps if fps > 0 else 0.0
        cap.release()

        t_start = time.time()
        capture = None
        try:
            demo = BaseDemo(
                video_path  = str(video_path),
                output_path = output_path,
                enable_reid = self.enable_reid,
            )
            try:
                from alphaction.config import cfg as _acfg
                _acfg.defrost()
            except Exception:
                pass

            video_writer = demo.make_visualizer()
            predictor    = demo.make_predictor()
            capture      = MetricsCapture(video_writer, condition, video_name)
            pred_done    = False
            frame_idx    = 0

            try:
                for frame_idx in _tqdm(
                    _count(),
                    desc=f"  [{condition}] {video_name}",
                    unit=" frame", leave=False,
                ):
                    if not predictor._thread.is_alive() and not pred_done:
                        print(f"\n  ❌ Predictor thread crashed on {video_name}"); break

                    with _torch.no_grad():
                        orig_img, boxes, scores, ids = predictor.read_track()

                    if orig_img is None:
                        predictor.signal_tracking_done(); break

                    video_writer.send_track((boxes, ids))
                    while not pred_done:
                        result = predictor.read_result()
                        if result is None: break
                        elif result == "done": pred_done = True
                        else: video_writer.send_result(result)

            except KeyboardInterrupt:
                print(f"\n  Interrupted during {video_name}.")

            video_writer.send_track("DONE")
            while not pred_done:
                result = predictor.read_result()
                if result is None: _sleep(0.05)
                elif result == "done": pred_done = True
                else: video_writer.send_result(result)

            video_writer.send_result("DONE")
            video_writer.show_progress(frame_idx)
            video_writer.close()
            predictor.terminate()

            vm.processing_time_s = time.time() - t_start
            vm.processing_fps    = frame_idx / vm.processing_time_s if vm.processing_time_s > 0 else 0.0
            capture.fill(vm)

        except Exception as e:
            vm.success = False; vm.error_msg = str(e)
            vm.processing_time_s = time.time() - t_start
            print(f"\n  ERROR in {video_name}: {e}"); traceback.print_exc()

        if vm.success:
            NotebookUtils.compare_videos(
                videos   = [
                    {"path": str(video_path), "label": f"[{condition.replace('_',' ').title()}]  {video_name}  — raw"},
                    {"path": output_path,     "label": f"[{condition.replace('_',' ').title()}]  {video_name}  — annotated"},
                ],
                n_frames = self.n_frames,
                title    = f"[{condition.replace('_',' ').title()}]  {video_name}",
            )

        return vm, (capture.prediction_events if capture and vm.success else [])

    def run(self) -> None:
        entries = self._discover()
        if not entries:
            print(f"No videos found under {self.video_dir}/")
            print("Expected: test_videos/{clean,crowd,fast_motion,occlusion,low_light}/*.mp4")
            return

        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"\n{'='*60}")
        print(f"  AlphAction Robustness Evaluation  —  {ts}")
        print(f"  {len(entries)} videos  |  {len({e[0] for e in entries})} conditions")
        print(f"{'='*60}\n")

        self.all_metrics = []
        self.all_events  = []

        for cond, vpath in entries:
            print(f"\n▶  [{cond}]  {vpath.name}")
            vm, events = self._run_one(cond, vpath)
            self.all_metrics.append(vm)
            self.all_events.extend(events)

            status = (f"✓  det={vm.detection_rate:.0%}  "
                      f"persons/f={vm.avg_persons_per_frame:.1f}  "
                      f"conf={vm.avg_confidence:.2f}  "
                      f"actions={vm.unique_actions_detected}  "
                      f"{vm.processing_fps:.1f}fps"
                      if vm.success else f"✗  {vm.error_msg}")
            print(f"   {status}")

        print(f"\n✅ Evaluation complete.")
        self._show_table()
        print("\nRun evaluator.plot() to see the charts.")

    def _show_table(self) -> None:
        if not self.all_metrics: return

        rows = []
        by_cond: Dict[str, List[VideoMetrics]] = {}
        for vm in self.all_metrics:
            by_cond.setdefault(vm.condition, []).append(vm)

        for cond in CONDITIONS:
            ok = [v for v in by_cond.get(cond, []) if v.success]
            if not ok: continue
            avg = lambda f: sum(f(v) for v in ok) / len(ok)
            rows.append({
                "Condition"       : cond.replace("_", " ").title(),
                "Videos"          : len(ok),
                "Det. Rate"       : f"{avg(lambda v: v.detection_rate):.1%}",
                "Avg Persons/f"   : f"{avg(lambda v: v.avg_persons_per_frame):.2f}",
                "Avg Confidence"  : f"{avg(lambda v: v.avg_confidence):.3f}",
                "Unique Actions"  : int(avg(lambda v: v.unique_actions_detected)),
                "Proc. FPS"       : f"{avg(lambda v: v.processing_fps):.1f}",
            })

        df = _pd.DataFrame(rows)
        _display(_HTML("<h4>📊 Robustness Evaluation — Summary by Condition</h4>"))
        _display(df.style
                  .set_properties(**{"text-align": "center"})
                  .set_table_styles([
                      {"selector": "th", "props": [("background-color", "#2d2d5e"),
                                                    ("color", "white"),
                                                    ("font-weight", "bold"),
                                                    ("text-align", "center")]},
                      {"selector": "tr:nth-child(even)", "props": [("background-color", "#f0f4ff")]},
                  ])
                  .pipe(lambda s: s.hide(axis="index") if hasattr(s, "hide") else s.hide_index()))

        clean_ok = [v for v in by_cond.get("clean", []) if v.success]
        if clean_ok:
            cavg = lambda f: sum(f(v) for v in clean_ok) / len(clean_ok)
            b_det  = cavg(lambda v: v.detection_rate)
            b_conf = cavg(lambda v: v.avg_confidence)
            deg_rows = []
            for cond in [c for c in CONDITIONS if c != "clean"]:
                ok = [v for v in by_cond.get(cond, []) if v.success]
                if not ok: continue
                oavg  = lambda f: sum(f(v) for v in ok) / len(ok)
                d_det  = oavg(lambda v: v.detection_rate) - b_det
                d_conf = oavg(lambda v: v.avg_confidence) - b_conf
                assess = ("🔴 Significant" if d_det < -0.20 else
                          "🟠 Moderate"    if d_det < -0.08 else
                          "🟡 Mild"         if d_det < 0     else
                          "🟢 None / Better")
                deg_rows.append({
                    "Condition"  : cond.replace("_"," ").title(),
                    "Det Rate Δ" : f"{d_det:+.1%}",
                    "Conf Δ"     : f"{d_conf:+.3f}",
                    "Assessment" : assess,
                })
            if deg_rows:
                _display(_HTML("<h4>📉 Degradation vs Clean Baseline</h4>"))
                _display(_pd.DataFrame(deg_rows).style
                            .pipe(lambda s: s.hide(axis="index") if hasattr(s, "hide") else s.hide_index())
                            .set_properties(**{"text-align":"center"})
                            .set_table_styles([
                                {"selector":"th","props":[("background-color","#2d2d5e"),
                                                           ("color","white"),
                                                           ("text-align","center")]},
                            ]))

    def plot(self) -> None:
        if not self.all_metrics:
            print("No results — run evaluator.run() first."); return

        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        import plotly.express as px

        by_cond: Dict[str, List[VideoMetrics]] = {}
        for vm in self.all_metrics:
            by_cond.setdefault(vm.condition, []).append(vm)

        conds  = [c for c in CONDITIONS if c in by_cond]
        labels = [c.replace("_"," ").title() for c in conds]
        colors = _COLORS[:len(conds)]

        def cavg(cond, f):
            ok = [v for v in by_cond[cond] if v.success]
            return sum(f(v) for v in ok) / len(ok) if ok else 0.0

        det_rates   = [cavg(c, lambda v: v.detection_rate)        for c in conds]
        avg_confs   = [cavg(c, lambda v: v.avg_confidence)        for c in conds]
        avg_persons = [cavg(c, lambda v: v.avg_persons_per_frame) for c in conds]
        proc_fps    = [cavg(c, lambda v: v.processing_fps)        for c in conds]

        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=("Detection Rate by Condition",
                            "Avg Action Confidence",
                            "Avg Persons per Frame",
                            "Processing Speed (FPS)"),
            vertical_spacing=0.18, horizontal_spacing=0.12,
        )
        def bar(vals, fmt, row, col):
            fig.add_trace(go.Bar(
                x=labels, y=vals, marker_color=colors,
                text=[fmt % v for v in vals], textposition="outside",
                showlegend=False,
            ), row=row, col=col)

        bar(det_rates,   "%.1f%%", 1, 1)
        bar(avg_confs,   "%.3f",   1, 2)
        bar(avg_persons, "%.2f",   2, 1)
        bar(proc_fps,    "%.1f",   2, 2)
        fig.update_yaxes(tickformat=".0%", row=1, col=1)
        fig.update_layout(
            height=620, width=920,
            title_text="<b>AlphAction Robustness Evaluation — Summary</b>",
            title_font_size=16, paper_bgcolor="white", plot_bgcolor="#f9f9f9",
            margin=dict(t=80, b=40, l=60, r=40),
        )
        fig.show()

        action_counts: Dict[str, Dict[str, int]] = {}
        for vm in self.all_metrics:
            if not vm.success: continue
            for action in (a.strip() for a in vm.actions_detected.split(",") if a.strip()):
                action_counts.setdefault(action, {})
                action_counts[action][vm.condition] = action_counts[action].get(vm.condition, 0) + 1

        if action_counts:
            all_actions = sorted(action_counts)
            z = [[action_counts[a].get(c, 0) for c in conds] for a in all_actions]
            fig2 = go.Figure(go.Heatmap(
                z=z, x=labels, y=all_actions, colorscale="Blues",
                text=z, texttemplate="%{text}", textfont_size=10,
            ))
            fig2.update_layout(
                title="<b>Action Detections per Condition</b>",
                title_font_size=15,
                height=max(350, 28 * len(all_actions)), width=700,
                margin=dict(t=60, b=60, l=180, r=40),
                xaxis_title="Condition", yaxis_title="Action",
                paper_bgcolor="white",
            )
            fig2.show()

        if self.all_events:
            df = _pd.DataFrame([asdict(e) for e in self.all_events])
            df["condition"] = df["condition"].str.replace("_"," ").str.title()
            fig3 = px.box(
                df, x="condition", y="confidence", color="condition",
                points="outliers", color_discrete_sequence=colors,
                title="<b>Action Confidence Distribution by Condition</b>",
                labels={"confidence":"Confidence Score","condition":"Condition"},
            )
            fig3.update_layout(height=420, width=720, showlegend=False,
                               paper_bgcolor="white", plot_bgcolor="#f9f9f9")
            fig3.show()

        print("\n✅ Charts rendered.")

print("✅ RobustnessEvaluator  ready")
print("   evaluator = RobustnessEvaluator(video_dir='test_videos')")
print("   evaluator.run()    # runs all videos, shows frame previews + table inline")
print("   evaluator.plot()   # renders Plotly charts")

#  👉 Thermal Camera Adaptation for Privacy-Preserving Scenarios
AlphAction was trained exclusively on RGB video from the AVA dataset (Gu et al., 2018)
Thermal infrared (IR) cameras produce single-channel greyscale images encoding emitted
heat rather than reflected light a significant domain shift.

## ⚙️ `ThermalAdapter` — Pseudo-Thermal Conversion

In [ ]:
from __future__ import annotations
import cv2
import numpy as np

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.cm as _cm
    _HAS_MPL = True
except ImportError:
    _HAS_MPL = False
    print("WARNING: matplotlib not found — using approximate INFERNO colourmap.")


class ThermalAdapter:
    """
    Converts an RGB/BGR frame to a pseudo-thermal representation.

    Stages
    ------
    1. Luminance extraction from CIE LAB
    2. Gaussian blur (sigma=2.0) — suppresses high-frequency texture
    3. INFERNO false-colour mapping — restores 3-channel structure
    4. CLAHE contrast normalisation — recovers AlphAction activation range
    """

    def __init__(self, sigma: float = 2.0, clahe_clip: float = 2.0, clahe_tile: int = 8):
        self.sigma = sigma
        self.clahe = cv2.createCLAHE(clipLimit=clahe_clip,
                                      tileGridSize=(clahe_tile, clahe_tile))
        ksize = int(6 * sigma + 1) | 1   # must be odd
        self.ksize = (ksize, ksize)

    def _luminance(self, bgr: np.ndarray) -> np.ndarray:
        """Stage 1+2: CIE-LAB luminance → Gaussian blur."""
        L = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)[:, :, 0]
        return cv2.GaussianBlur(L, self.ksize, self.sigma)

    def _inferno(self, blurred: np.ndarray) -> np.ndarray:
        """Stage 3: map blurred uint8 → BGR via INFERNO colourmap."""
        normed = blurred.astype(np.float32) / 255.0
        if _HAS_MPL:
            rgba = _cm.inferno(normed)
            rgb  = (rgba[:, :, :3] * 255).astype(np.uint8)
        else:
            r = np.clip(normed * 2.0,        0, 1)
            g = np.clip(normed * 1.3 - 0.3,  0, 1)
            b = np.zeros_like(normed)
            rgb = (np.stack([r, g, b], axis=2) * 255).astype(np.uint8)
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

    def convert(self, bgr: np.ndarray) -> np.ndarray:
        """Full pipeline: Luminance → Blur → INFERNO → CLAHE  (Table 4.5 row 4)."""
        bgr_inferno = self._inferno(self._luminance(bgr))
        result = bgr_inferno.copy()
        for c in range(3):
            result[:, :, c] = self.clahe.apply(bgr_inferno[:, :, c])
        return result

    def convert_greyscale(self, bgr: np.ndarray) -> np.ndarray:
        """Greyscale replicated to 3 channels  (Table 4.5 row 2)."""
        grey = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        return cv2.merge([grey, grey, grey])

    def convert_inferno_no_clahe(self, bgr: np.ndarray) -> np.ndarray:
        """Luminance + INFERNO, no CLAHE  (Table 4.5 row 3)."""
        return self._inferno(self._luminance(bgr))

    def process_video(
        self,
        in_path:  str,
        out_path: str,
        mode:     str = "adapted",  # "greyscale" | "inferno" | "adapted"
    ) -> None:
        """Convert an entire video file to the chosen pseudo-thermal mode."""
        cap = cv2.VideoCapture(in_path)
        if not cap.isOpened():
            raise FileNotFoundError(f"Cannot open video: {in_path}")

        fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
        w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(out_path, fourcc, fps, (w, h))

        converters = {
            "greyscale": self.convert_greyscale,
            "inferno":   self.convert_inferno_no_clahe,
            "adapted":   self.convert,
        }
        fn = converters.get(mode)
        if fn is None:
            cap.release(); writer.release()
            raise ValueError(f"Unknown mode '{mode}'. Choose: greyscale | inferno | adapted")

        from tqdm import tqdm
        for _ in tqdm(range(total), desc=f"  [{mode}] converting"):
            ok, frame = cap.read()
            if not ok:
                break
            writer.write(fn(frame))

        cap.release(); writer.release()
        print(f"  ✅ Written → {out_path}")


print("✅ ThermalAdapter ready.")
print("   adapter = ThermalAdapter()")
print("   thermal_frame = adapter.convert(bgr_frame)         # full pipeline")
print("   adapter.process_video('in.mp4', 'out.mp4', 'adapted')")


In [ ]:
import os
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image as _Img
from pathlib import Path

os.chdir(str(DEMO_DIR))

_SAMPLE_VIDEO = "test_video_3s_480p.mp4"
_N_PREVIEW    = 4          # frames to show

def _sample_bgr_frames(video_path: str, n: int = 4):
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step  = max(1, total // n)
    frames = []
    for i in range(n):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ok, f = cap.read()
        if ok:
            frames.append(f)
    cap.release()
    return frames

if not os.path.exists(_SAMPLE_VIDEO):
    print(f"⚠️  {_SAMPLE_VIDEO} not found — skipping frame preview.")
else:
    _adapter = ThermalAdapter()
    _frames  = _sample_bgr_frames(_SAMPLE_VIDEO, _N_PREVIEW)

    _modes = [
        ("RGB (original)",                    lambda f: cv2.cvtColor(f, cv2.COLOR_BGR2RGB)),
        ("Greyscale (3-ch)",                  lambda f: cv2.cvtColor(_adapter.convert_greyscale(f), cv2.COLOR_BGR2RGB)),
        ("Luminance + INFERNO (no CLAHE)",    lambda f: cv2.cvtColor(_adapter.convert_inferno_no_clahe(f), cv2.COLOR_BGR2RGB)),
        ("Luminance + INFERNO + CLAHE ✅",    lambda f: cv2.cvtColor(_adapter.convert(f), cv2.COLOR_BGR2RGB)),
    ]

    fig, axes = plt.subplots(len(_modes), _N_PREVIEW,
                             figsize=(4 * _N_PREVIEW, 3.2 * len(_modes)),
                             facecolor="#0d0d1a")
    for r, (label, fn) in enumerate(_modes):
        for c, bgr in enumerate(_frames):
            ax = axes[r][c]
            ax.imshow(fn(bgr))
            ax.axis("off")
            if c == 0:
                ax.set_ylabel(label, fontsize=9, color="white",
                              rotation=0, labelpad=180, va="center")

    plt.suptitle("Pseudo-Thermal Conversion — Frame Comparison",
                 color="white", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    _preview_path = str(FIGURES_DIR / "thermal_frame_comparison.png")
    plt.savefig(_preview_path, dpi=110, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    display(_Img(_preview_path))
    print(f"  ✅ Saved → {_preview_path}")


In [ ]:
import os
from pathlib import Path

os.chdir(str(DEMO_DIR))
_adapter = ThermalAdapter()

_INPUT_VIDEO = "test_video_3s_480p.mp4"
_THERMAL_VIDEOS = {
    "greyscale": "test_video_greyscale.mp4",
    "inferno":   "test_video_inferno.mp4",
    "adapted":   "test_video_thermal.mp4",
}

if not os.path.exists(_INPUT_VIDEO):
    print(f"⚠️  {_INPUT_VIDEO} not found — skipping video conversion.")
    print("   Place a short test video in the demo/ folder and re-run.")
else:
    for mode, out_name in _THERMAL_VIDEOS.items():
        if os.path.exists(out_name):
            print(f"  ↩  {out_name} already exists — skipping.")
        else:
            _adapter.process_video(_INPUT_VIDEO, out_name, mode=mode)

    # Quick side-by-side frame check
    NotebookUtils.compare_videos(
        videos=[
            {"path": _INPUT_VIDEO,               "label": "RGB (original)"},
            {"path": _THERMAL_VIDEOS["greyscale"],"label": "Greyscale (3-ch)"},
            {"path": _THERMAL_VIDEOS["inferno"],  "label": "Luminance + INFERNO"},
            {"path": _THERMAL_VIDEOS["adapted"],  "label": "INFERNO + CLAHE ✅"},
        ],
        n_frames=5,
        title="Pseudo-Thermal Conversion — Video Comparison",
    )


## ▶️ Run AlphAction on the Thermal-Adapted Video

Pass the adapted video through the full pipeline to collect action confidence scores  
for the *Avg Confidence* column of **Table 4.5**.


In [ ]:
import os

os.chdir(str(DEMO_DIR))

_thermal_input  = _THERMAL_VIDEOS["adapted"]
_thermal_output = "output_thermal_adapted.mp4"

if not os.path.exists(_thermal_input):
    print(f"⚠️  {_thermal_input} not found — run the conversion cell first.")
else:
    _demo_thermal = BaseDemo(
        video_path       = _thermal_input,
        output_path      = _thermal_output,
        enable_reid      = False,
        visual_threshold = 0.5,
        detect_rate      = 4,
    )
    _demo_thermal.run()

    if os.path.exists(_thermal_output):
        NotebookUtils.show_video(_thermal_output)
        NotebookUtils.compare_videos(
            videos=[
                {"path": _thermal_input,  "label": "Thermal Input"},
                {"path": _thermal_output, "label": "AlphAction Predictions"},
            ],
            n_frames=5,
            title="AlphAction — Thermal Adaptation Output",
        )


## 📊 FID Evaluation — Domain Gap Quantification

**Fréchet Inception Distance (FID)** measures the distributional distance between  
Inception-v3 pool3 features of RGB frames and each converted version.  
**Lower FID = smaller domain gap from AlphAction's RGB training distribution.**

This replicates the analysis in `thermal_eval.py` inline so results appear directly  
in the notebook and feed into Table 4.5.


In [ ]:
import csv
import time
from pathlib import Path
from typing import Dict, List

import cv2
import numpy as np
import torch
import torchvision.models as models
import torchvision.transforms as T
from IPython.display import display, HTML as _HTML
import pandas as _pd


class _InceptionExtractor:
    def __init__(self, device):
        self.device = device
        m = models.inception_v3(pretrained=True, transform_input=False)
        self.model = torch.nn.Sequential(
            m.Conv2d_1a_3x3, m.Conv2d_2a_3x3, m.Conv2d_2b_3x3,
            torch.nn.MaxPool2d(3, stride=2),
            m.Conv2d_3b_1x1, m.Conv2d_4a_3x3,
            torch.nn.MaxPool2d(3, stride=2),
            m.Mixed_5b, m.Mixed_5c, m.Mixed_5d,
            m.Mixed_6a, m.Mixed_6b, m.Mixed_6c, m.Mixed_6d, m.Mixed_6e,
            m.Mixed_7a, m.Mixed_7b, m.Mixed_7c,
            torch.nn.AdaptiveAvgPool2d((1, 1)),
            torch.nn.Flatten(),
        ).eval().to(device)
        self.tf = T.Compose([
            T.ToPILImage(), T.Resize((299, 299)), T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),
        ])

    @torch.no_grad()
    def extract(self, bgr_frames: List[np.ndarray], batch: int = 32) -> np.ndarray:
        out = []
        for i in range(0, len(bgr_frames), batch):
            chunk = bgr_frames[i:i+batch]
            imgs  = torch.stack([
                self.tf(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in chunk
            ]).to(self.device)
            out.append(self.model(imgs).cpu().numpy())
        return np.concatenate(out, 0).astype(np.float32)


def _fid(a: np.ndarray, b: np.ndarray) -> float:
    """Stable FID via eigendecomposition (no scipy needed)."""
    mu_a, mu_b = a.mean(0), b.mean(0)
    sig_a = np.cov(a, rowvar=False) + np.eye(a.shape[1]) * 1e-6
    sig_b = np.cov(b, rowvar=False) + np.eye(b.shape[1]) * 1e-6
    vals, vecs = np.linalg.eigh(sig_a @ sig_b)
    sqrt_prod = vecs @ np.diag(np.sqrt(np.maximum(vals, 0))) @ vecs.T
    score = float((mu_a - mu_b) @ (mu_a - mu_b) + np.trace(sig_a + sig_b - 2*sqrt_prod))
    return max(0.0, score)


def _sample_frames(video_paths: List[str], n_per_video: int = 60) -> List[np.ndarray]:
    frames = []
    for vp in video_paths:
        cap   = cv2.VideoCapture(vp)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release(); continue
        step = max(1, total // n_per_video)
        for fi in range(0, total, step):
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ok, fr = cap.read()
            if ok:
                frames.append(cv2.resize(fr, (299, 299)))
        cap.release()
    return frames


def run_fid_evaluation(
    video_paths: List[str],
    n_per_video: int = 60,
) -> Dict[str, float]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Device: {device}")

    print(f"  Sampling frames from {len(video_paths)} video(s)…")
    rgb_frames = _sample_frames(video_paths, n_per_video)
    if not rgb_frames:
        raise ValueError("No frames extracted — check video paths.")
    print(f"  {len(rgb_frames)} frames sampled.")

    _adp = ThermalAdapter()
    grey_frames    = [_adp.convert_greyscale(f)        for f in rgb_frames]
    inferno_frames = [_adp.convert_inferno_no_clahe(f) for f in rgb_frames]
    thermal_frames = [_adp.convert(f)                  for f in rgb_frames]

    print("  Extracting Inception-v3 features (≈1–2 min)…")
    ext = _InceptionExtractor(device)
    f_rgb     = ext.extract(rgb_frames)
    f_grey    = ext.extract(grey_frames)
    f_inferno = ext.extract(inferno_frames)
    f_thermal = ext.extract(thermal_frames)

    print("  Computing FID scores…")
    half = len(f_rgb) // 2
    return {
        "RGB (self, reference)":                    round(_fid(f_rgb[:half], f_rgb[half:]) if half > 1 else 0.0, 2),
        "Greyscale (3-ch replicated)":              round(_fid(f_rgb, f_grey),    2),
        "Luminance + INFERNO (no CLAHE)":           round(_fid(f_rgb, f_inferno), 2),
        "Luminance + INFERNO + CLAHE (ThermalAdapter)": round(_fid(f_rgb, f_thermal), 2),
    }


print("✅ FID utilities loaded.")
print("   Call run_fid_evaluation(['video1.mp4', ...]) to compute scores.")


In [ ]:
import os
import time
from pathlib import Path
from IPython.display import display, HTML as _HTML
import pandas as _pd

os.chdir(str(DEMO_DIR))

# Use the test video (duplicated for a stable estimate when only 1 video is available)
_FID_VIDEO = "test_video_3s_480p.mp4"

if not os.path.exists(_FID_VIDEO):
    print(f"⚠️  {_FID_VIDEO} not found — place a test video in demo/ and re-run.")
else:
    t0 = time.time()
    _fid_scores = run_fid_evaluation(
        video_paths = [_FID_VIDEO, _FID_VIDEO],   # duplicate for stable covariance estimate
        n_per_video = 60,
    )
    _elapsed = time.time() - t0

    _out_dir = Path("thermal_results")
    _out_dir.mkdir(exist_ok=True)
    _csv_path = str(_out_dir / "thermal_fid.csv")
    import csv
    with open(_csv_path, "w", newline="") as _f:
        _w = csv.writer(_f)
        _w.writerow(["Method", "FID"])
        for _m, _v in _fid_scores.items():
            _w.writerow([_m, _v])
    print(f"  CSV saved → {_csv_path}")

    _rows = [{"Method": m, "FID ↓": v} for m, v in _fid_scores.items()]
    _df   = _pd.DataFrame(_rows)

    display(_HTML("<h4>📊 Table 4.5 — Pseudo-Thermal Adaptation: FID vs RGB Baseline</h4>"))
    display(_df.style
        .set_properties(**{"text-align": "left"})
        .set_table_styles([
            {"selector": "th", "props": [("background-color","#2d2d5e"),
                                          ("color","white"),("font-weight","bold")]},
            {"selector": "tr:nth-child(even)", "props": [("background-color","#f0f4ff")]},
        ])
        .pipe(lambda s: s.hide(axis="index") if hasattr(s, "hide") else s.hide_index())
        .highlight_min(subset=["FID ↓"], color="#c6efce")
        .format({"FID ↓": "{:.2f}"})
    )

    print(f"\n  ✅ FID evaluation complete in {_elapsed:.1f}s."

## 📈 FID Bar Chart — Domain Gap Visualisation

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Image as _Img
from pathlib import Path

try:
    _labels = list(_fid_scores.keys())
    _values = list(_fid_scores.values())
    _short  = [
        "RGB\n(self)",
        "Greyscale\n(3-ch)",
        "INFERNO\n(no CLAHE)",
        "INFERNO\n+ CLAHE ✅",
    ]
    _colours = ["#4C78A8", "#E45756", "#F58518", "#54A24B"]

    fig, ax = plt.subplots(figsize=(9, 4.5), facecolor="#0d0d1a")
    ax.set_facecolor("#0d0d1a")
    bars = ax.bar(_short, _values, color=_colours, width=0.5, zorder=3)

    for bar, val in zip(bars, _values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(_values)*0.01,
                f"{val:.1f}", ha="center", va="bottom", color="white", fontsize=11, fontweight="bold")

    ax.set_ylabel("FID  (lower = smaller domain gap)", color="white", fontsize=11)
    ax.set_title("Pseudo-Thermal Adaptation — FID vs RGB Baseline",
                 color="white", fontsize=13, fontweight="bold", pad=14)
    ax.tick_params(colors="white")
    for spine in ax.spines.values():
        spine.set_edgecolor("#444")
    ax.yaxis.grid(True, linestyle="--", alpha=0.3, color="grey", zorder=0)
    ax.set_ylim(0, max(_values) * 1.18)

    plt.tight_layout()
    _fig_path = str(FIGURES_DIR / "thermal_fid_bar.png")
    plt.savefig(_fig_path, dpi=120, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    display(_Img(_fig_path))
    print(f"  ✅ Chart saved → {_fig_path}")

except NameError:
    print("⚠️  _fid_scores not found — run the FID evaluation cell first.")


# 👉 Appendix A — Pipeline Quick Reference

```bash
# Activate environment
conda activate alphaction_a100
cd ~/Long-Term-Multi-Person-Action-Recognition-and-Tracking/AlphactionFramework/demo

# Run demo (with face recognition Re-ID — default)
python3 demo.py \
    --video-path  test_video.mp4 \
    --output-path output.mp4 \
    --cfg-path    ../config_files/resnet101_8x8f_denseserial.yaml \
    --weight-path ../data/models/aia_models/resnet101_8x8f_denseserial.pth

# Run demo with fine-tuned Re-ID weights
python3 demo.py \
    --video-path   test_video.mp4 \
    --reid-weights finetuned_reid.pth \
    --output-path  output_finetuned.mp4

# Run ablation study (Configs 0-5 including face recognition)
python3 ablation_config.py

# Convert video to thermal and run pipeline
python3 -c "
from thermal_adapter import ThermalAdapter
ThermalAdapter().process_video('test.mp4','test_thermal.mp4','adapted')
"
python3 demo.py --video-path test_thermal.mp4 --output-path output_thermal.mp4
```

## ⚙️ References

1. Gu et al. (2018). AVA: A Video Dataset of Spatio-Temporally Localized Atomic Actions. *CVPR*.
2. Feichtenhofer et al. (2019). SlowFast Networks for Video Recognition. *ICCV*.
3. Tang et al. (2020). Asynchronous Interaction Aggregation for Action Detection. *ECCV* (AlphAction).
4. **Deng et al. (2019). ArcFace: Additive Angular Margin Loss. *CVPR*.**
5. **Deng et al. (2020). RetinaFace: Single-Stage Dense Face Localisation. *CVPR*.**
6. Selvaraju et al. (2017). Grad-CAM: Visual Explanations from Deep Networks. *ICCV*.
7. Gal & Ghahramani (2016). Dropout as a Bayesian Approximation. *ICML*.
8. Zheng et al. (2015). Scalable Person Re-identification. *ICCV*.
9. Gade & Moeslund (2014). Thermal Cameras and Applications. *MVA*.
10. Kalogeiton et al. (2017). Action Tubelet Detector. *ICCV*.
11. Wu et al. (2019). Long-Term Feature Banks. *CVPR*.
12. Wang et al. (2018). Non-Local Neural Networks. *CVPR*.
13. Girshick (2015). Fast R-CNN. *ICCV*.
14. Arnab et al. (2021). ViViT: Video Vision Transformer. *ICCV*.
15. Feichtenhofer et al. (2018). SlowFast Networks. *ICCV*.
16. Munir et al. (2021). Thermal Action Recognition. *IEEE Access*.
17. He et al. (2016). Deep Residual Learning. *CVPR*.
18. Liu et al. (2021). Group Activity Recognition. *ICCV*.
19. Cheng et al. (2020). Faster Person Re-ID. *ECCV*.
20. Wen et al. (2021). SLAK: Slimmable Attention. *CVPR*.

---

<p align="center">
  <img src="https://res.cloudinary.com/dhqdentd8/image/upload/v1752880345/closing_image_yjg6dq.jpg" width="100%" alt="Closing"/>
</p>
